# GraphWorld-Ads
## Counterfactual Graph World Models for Autonomous Advertising Optimization

**Research prototype:** heterogeneous temporal Graph Transformer + counterfactual rollouts + uncertainty-aware Graph-MPC + entropy-regularized optimal transport + multi-agent advertiser dynamics.

### Data policy
- **Real:** Amazon ESCI query–product relevance, product metadata, brand and locale structure.
- **Optional real calibration:** Criteo attribution logs for click/conversion/cost marginal sanity checks.
- **Semi-synthetic:** advertiser IDs, campaigns, bids, budgets, auction competition, interventions and future campaign states.

The public Amazon Shopping Queries Dataset does **not** contain advertiser auction logs, so simulated outcomes are never presented as observed Amazon Ads outcomes.

### Research question
Given a heterogeneous marketplace graph \(G_t\) and advertiser intervention \(a_t\), can a learned graph world model forecast \(G_{t+1:t+H}\) well enough to plan safe budget/bid actions under competition and uncertainty?

`Amazon ESCI → heterogeneous ads graph → custom temporal relation-aware GTN → probabilistic world model → counterfactual rollouts → OT budget allocator → Graph-MPC → safety gate → agent tools`

In [ ]:
# Colab setup
!pip -q install "datasets>=3.0.0" "huggingface_hub>=0.25.0" pyarrow scikit-learn tqdm

import os, math, json, random, hashlib, warnings
from dataclasses import dataclass
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from datasets import load_dataset
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
SEED=17
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Defaults are deliberately Colab-safe. Set True for the larger paper-style run.
FULL_RESEARCH_RUN=False
if FULL_RESEARCH_RUN:
    MAX_ESCI_ROWS=60_000; MAX_QUERIES=4_000; MAX_PRODUCTS=12_000; MAX_CAMPAIGNS=900; TIME_STEPS=20
else:
    MAX_ESCI_ROWS=20_000; MAX_QUERIES=800; MAX_PRODUCTS=2_500; MAX_CAMPAIGNS=240; TIME_STEPS=10
USE_CRITEO_CALIBRATION=False
print("device",DEVICE,"full_research_run",FULL_RESEARCH_RUN)

## 1. Public datasets

### Amazon ESCI Shopping Queries
Used for real Amazon query↔product relevance, product metadata, brands and locales (US/ES/JP). The notebook streams a Hugging Face mirror rather than downloading the full multi-GB dataset.

### Optional Criteo Attribution Dataset
Used only for marginal click/conversion/cost sanity checks. It is **not** joined to ESCI row-by-row and is never described as Amazon Ads data.

In [ ]:
ESCI_LABEL_SCORE={"Exact":1.0,"E":1.0,"Substitute":.72,"S":.72,"Complement":.48,"C":.48,"Irrelevant":.05,"I":.05}
ESCI_REQUIRED={"query","query_id","product_id","product_locale","esci_label","product_title","product_brand"}
ESCI_ALLOWED_LABELS=set(ESCI_LABEL_SCORE)
ESCI_ALLOWED_LOCALES={"us","es","jp"}

def stable_keep(*parts, modulo=1000, threshold=220):
    h=int(hashlib.sha1("||".join(map(str,parts)).encode()).hexdigest()[:12],16)
    return h%modulo < threshold

def _validate_esci_record(r):
    missing=ESCI_REQUIRED-set(r.keys())
    if missing: raise ValueError(f"ESCI schema mismatch; missing columns: {sorted(missing)}")
    if str(r["product_locale"]) not in ESCI_ALLOWED_LOCALES:
        return False
    lab=str(r["esci_label"])
    if lab not in ESCI_ALLOWED_LABELS:
        raise ValueError(f"Unexpected ESCI label {lab!r}; refusing silent remap")
    return True

def load_esci_sample(max_rows=MAX_ESCI_ROWS):
    ds=load_dataset("tasksource/esci", split="train", streaming=True)
    # IterableDataset exposes features in the current HF mirror; row-level validation remains authoritative.
    if getattr(ds,"features",None):
        cols=set(ds.features.keys()); miss=ESCI_REQUIRED-cols
        if miss: raise ValueError(f"ESCI dataset features missing {sorted(miss)}")
    rows=[]
    for r in tqdm(ds,total=max_rows,desc="Streaming ESCI"):
        if not _validate_esci_record(r): continue
        if not stable_keep(r.get("query_id"),r.get("product_id"),r.get("product_locale")): continue
        product_text=str(r.get("product_text") or r.get("product_description") or r.get("product_title") or "")
        rows.append({
            "query_raw_id":str(r["query_id"]),"query":str(r["query"]),
            "product_raw_id":str(r["product_id"]),"locale":str(r["product_locale"]),
            "label":str(r["esci_label"]),"title":str(r.get("product_title") or ""),
            "brand":str(r.get("product_brand") or "UNKNOWN_BRAND"),"product_text":product_text,
        })
        if len(rows)>=max_rows: break
    df=pd.DataFrame(rows)
    if df.empty: raise RuntimeError("ESCI stream returned no usable rows")
    df["rel"]=df["label"].map(ESCI_LABEL_SCORE)
    if df["rel"].isna().any(): raise ValueError("Unmapped ESCI labels detected")
    return df

try:
    esci_raw=load_esci_sample(); print("real ESCI rows",len(esci_raw))
except Exception as e:
    print("ESCI unavailable; deterministic schema-faithful mock mode",repr(e))
    rng=np.random.default_rng(SEED); rows=[]
    for q in range(600):
        loc=["us","es","jp"][q%3]
        for _ in range(10):
            p=int(rng.integers(0,1800)); score=float(rng.beta(2,2))
            lab="E" if score>.8 else "S" if score>.55 else "C" if score>.3 else "I"
            rows.append(dict(query_raw_id=str(q),query=f"query {q}",product_raw_id=f"P{p}",locale=loc,label=lab,
                             title=f"product {p}",brand=f"brand_{p%120}",product_text=f"product {p} brand_{p%120}",
                             rel=ESCI_LABEL_SCORE[lab]))
    esci_raw=pd.DataFrame(rows)

required_internal={"query_raw_id","query","product_raw_id","locale","label","title","brand","product_text","rel"}
assert required_internal.issubset(esci_raw.columns)
assert set(esci_raw.locale.unique()).issubset(ESCI_ALLOWED_LOCALES)
assert esci_raw["rel"].between(0,1).all()
esci_raw.head()

In [ ]:
# Locale-scoped keys avoid assuming raw query/product IDs are globally unique across marketplaces.
esci_raw=esci_raw.copy()
esci_raw["query_id"]=esci_raw["locale"].astype(str)+"::"+esci_raw["query_raw_id"].astype(str)
esci_raw["product_id"]=esci_raw["locale"].astype(str)+"::"+esci_raw["product_raw_id"].astype(str)

top_q=esci_raw.query_id.value_counts().head(MAX_QUERIES).index
esci=esci_raw[esci_raw.query_id.isin(top_q)].copy()
top_p=esci.product_id.value_counts().head(MAX_PRODUCTS).index
esci=esci[esci.product_id.isin(top_p)].copy()
esci=(esci.sort_values(["query_id","rel"],ascending=[True,False])
           .drop_duplicates(["locale","query_id","product_id"])
           .reset_index(drop=True))
assert esci[["query_id","product_id","locale","query","title","brand"]].notna().all().all()
assert esci.duplicated(["locale","query_id","product_id"]).sum()==0
assert esci.groupby("query_id").locale.nunique().max()==1
assert esci.groupby("product_id").locale.nunique().max()==1
print({"rows":len(esci),"queries":esci.query_id.nunique(),"products":esci.product_id.nunique(),"brands":esci.brand.nunique(),"locales":esci.locale.value_counts().to_dict()})

In [ ]:
criteo_marginals=None
if USE_CRITEO_CALIBRATION:
    ds=load_dataset("criteo/criteo-attribution-dataset",split="train",streaming=True)
    required={"timestamp","campaign","click","conversion","cost"}
    if getattr(ds,"features",None):
        miss=required-set(ds.features.keys())
        if miss: raise ValueError(f"Criteo schema mismatch; missing {sorted(miss)}")
    rows=[]
    for r in tqdm(ds,total=50_000,desc="Criteo calibration"):
        miss=required-set(r.keys())
        if miss: raise ValueError(f"Criteo row missing {sorted(miss)}")
        rows.append({"click":int(r["click"]),"conversion":int(r["conversion"]),"cost":float(r["cost"])})
        if len(rows)>=50_000: break
    cr=pd.DataFrame(rows)
    assert cr.click.isin([0,1]).all() and cr.conversion.isin([0,1]).all() and np.isfinite(cr.cost).all()
    criteo_marginals={"ctr":float(cr.click.mean()),"cvr_per_impression":float(cr.conversion.mean()),
                      "cost_p50_transformed":float(cr.cost.median()),"cost_p90_transformed":float(cr.cost.quantile(.9))}
    print(criteo_marginals)
else:
    print("Criteo calibration disabled; campaign dynamics remain explicitly semi-synthetic.")

## 2. Heterogeneous ads graph

Node types: advertiser/brand, campaign, query, product, marketplace.
Relations: advertiser→campaign, campaign→product, campaign→query, query→product, product→advertiser, product→market, query→market.

In [ ]:
brands=esci.brand.fillna("UNKNOWN_BRAND").astype(str)
eligible=brands.value_counts(); eligible=eligible[eligible>=3].index.tolist()[:MAX_CAMPAIGNS]
prod_tbl=(esci[esci.brand.isin(eligible)].sort_values("rel",ascending=False).drop_duplicates("product_id")
          [["product_id","product_raw_id","brand","locale","title","product_text"]].reset_index(drop=True))
query_tbl=(esci[esci.product_id.isin(prod_tbl.product_id)].drop_duplicates("query_id")
           [["query_id","query_raw_id","query","locale"]].reset_index(drop=True))
advertiser_tbl=pd.DataFrame({"advertiser_id":sorted(prod_tbl.brand.unique())})
campaign_seed=prod_tbl.groupby(["brand","locale"],as_index=False).agg(product_count=("product_id","nunique"))
campaign_seed=campaign_seed.sort_values(["product_count","brand","locale"],ascending=[False,True,True]).head(MAX_CAMPAIGNS).reset_index(drop=True)
campaign_seed["campaign_id"]=[f"C{i}" for i in range(len(campaign_seed))]; campaign_seed["advertiser_id"]=campaign_seed.brand
market_tbl=pd.DataFrame({"locale":sorted(esci.locale.unique())})
if len(campaign_seed)==0 or len(query_tbl)==0 or len(prod_tbl)==0:
    raise RuntimeError("Sampling produced an empty campaign/query/product graph; increase MAX_ESCI_ROWS or sampling threshold")
print({"advertisers":len(advertiser_tbl),"campaigns":len(campaign_seed),"queries":len(query_tbl),"products":len(prod_tbl),"markets":len(market_tbl)})

In [ ]:
NODE_TYPES={"advertiser":0,"campaign":1,"query":2,"product":3,"market":4}
REL_TYPES={"adv_campaign":0,"campaign_product":1,"campaign_query":2,"query_product":3,"product_adv":4,"product_market":5,"query_market":6}
REV_OFFSET=len(REL_TYPES)
offset=0; node_maps={}
for name,vals in [("advertiser",advertiser_tbl.advertiser_id.astype(str).tolist()),("campaign",campaign_seed.campaign_id.tolist()),
                  ("query",query_tbl.query_id.astype(str).tolist()),("product",prod_tbl.product_id.astype(str).tolist()),
                  ("market",market_tbl.locale.astype(str).tolist())]:
    if len(vals)!=len(set(vals)): raise ValueError(f"Duplicate {name} node keys")
    node_maps[name]={v:offset+i for i,v in enumerate(vals)}; offset+=len(vals)
N=offset; node_type=np.zeros(N,dtype=np.int64)
for nt,mp in node_maps.items():
    for gid in mp.values(): node_type[gid]=NODE_TYPES[nt]
edge_set=set()
def add_bidir(s,d,rel):
    edge_set.add((int(s),int(d),REL_TYPES[rel])); edge_set.add((int(d),int(s),REL_TYPES[rel]+REV_OFFSET))
for r in campaign_seed.itertuples(): add_bidir(node_maps["advertiser"][str(r.advertiser_id)],node_maps["campaign"][r.campaign_id],"adv_campaign")
es=esci[esci.product_id.isin(prod_tbl.product_id)].copy(); camp_lookup={(r.brand,r.locale):r.campaign_id for r in campaign_seed.itertuples()}
for r in prod_tbl.itertuples():
    cid=camp_lookup.get((r.brand,r.locale))
    if cid is not None: add_bidir(node_maps["campaign"][cid],node_maps["product"][str(r.product_id)],"campaign_product")
for r in es.itertuples():
    q=node_maps["query"].get(str(r.query_id)); p=node_maps["product"].get(str(r.product_id))
    if q is not None and p is not None:
        add_bidir(q,p,"query_product"); cid=camp_lookup.get((r.brand,r.locale))
        if cid is not None: add_bidir(node_maps["campaign"][cid],q,"campaign_query")
for r in prod_tbl.itertuples():
    p=node_maps["product"][str(r.product_id)]; a=node_maps["advertiser"].get(str(r.brand)); m=node_maps["market"].get(str(r.locale))
    if a is not None: add_bidir(p,a,"product_adv")
    if m is not None: add_bidir(p,m,"product_market")
for r in query_tbl.itertuples():
    q=node_maps["query"][str(r.query_id)]; m=node_maps["market"].get(str(r.locale))
    if m is not None: add_bidir(q,m,"query_market")
edges=sorted(edge_set)
edge_index=torch.tensor([[s for s,d,r in edges],[d for s,d,r in edges]],dtype=torch.long)
edge_type=torch.tensor([r for s,d,r in edges],dtype=torch.long); num_relations=len(REL_TYPES)*2
assert edge_index.shape[1]==len(edge_set) and edge_index.max().item()<N
print("nodes",N,"deduplicated edges",edge_index.shape[1],"relations",num_relations)

## 3. Controlled multi-agent ads simulator

Real ESCI relevance anchors the marketplace graph. Bids, budgets, auctions, impressions, clicks, conversions and advertiser actions are semi-synthetic so counterfactual ground truth is known.

In [ ]:
campaign_ids=campaign_seed.campaign_id.tolist(); query_ids=query_tbl.query_id.tolist(); C,Q=len(campaign_ids),len(query_ids)
cid_to_i={c:i for i,c in enumerate(campaign_ids)}; qid_to_j={q:j for j,q in enumerate(query_ids)}
rel_cq=np.zeros((C,Q),np.float32)
for r in es.itertuples():
    cid=camp_lookup.get((r.brand,r.locale))
    if cid in cid_to_i and r.query_id in qid_to_j:
        i,j=cid_to_i[cid],qid_to_j[r.query_id]; rel_cq[i,j]=max(rel_cq[i,j],float(r.rel))
rel_cq[rel_cq<.25]=0
rng=np.random.default_rng(SEED); query_base_demand=rng.lognormal(5,.55,Q).astype(np.float32)
campaign_quality=rng.uniform(.65,1.15,C).astype(np.float32); campaign_value=rng.lognormal(4,.35,C).astype(np.float32)
@dataclass
class MarketState:
    budget:np.ndarray; bid:np.ndarray; spend:np.ndarray; impressions:np.ndarray; clicks:np.ndarray; conversions:np.ndarray
    revenue:np.ndarray; demand:np.ndarray; t:int; cum_revenue:np.ndarray; period_spend:np.ndarray

def initial_state():
    budget=rng.lognormal(8.3,.35,C).astype(np.float32); bid=rng.uniform(.45,2.1,C).astype(np.float32); z=np.zeros(C,np.float32)
    return MarketState(budget,bid,z.copy(),z.copy(),z.copy(),z.copy(),z.copy(),query_base_demand.copy(),0,z.copy(),z.copy())
def clone_state(s): return MarketState(**{k:(v.copy() if isinstance(v,np.ndarray) else v) for k,v in s.__dict__.items()})
def safe_ratio(a,b): return a/np.maximum(b,1e-6)
def market_step(state,action,stochastic=True):
    s=clone_state(state); bd=np.asarray(action["bid_delta"],np.float32); bud=np.asarray(action["budget_delta"],np.float32)
    if bd.shape!=(C,) or bud.shape!=(C,): raise ValueError("Action vector shape mismatch")
    s.bid=np.clip(s.bid*np.exp(bd),.15,8)
    proposed=np.maximum(50,s.budget*(1+bud)); s.budget=np.maximum(proposed,s.spend)  # cannot set budget below money already spent
    demand=np.maximum(1,s.demand*(1+.1*np.sin(2*np.pi*(s.t%7)/7)))
    strength=(rel_cq*campaign_quality[:,None])*np.sqrt(s.bid[:,None]+1e-4); strength=np.where(rel_cq>0,strength,-20)
    mx=strength.max(0,keepdims=True); ex=np.exp(np.clip(strength-mx,-30,30))*(rel_cq>0); share=ex/np.maximum(ex.sum(0,keepdims=True),1e-8)
    imp=(share*demand[None,:]).sum(1); avg_rel=(share*rel_cq).sum(1)/np.maximum(share.sum(1),1e-6); competition=max((rel_cq>0).sum(0).mean(),1)
    ctr=1/(1+np.exp(-(-3+2.3*avg_rel+.18*np.log1p(s.bid)))); cvr=1/(1+np.exp(-(-3.6+1.6*avg_rel+.35*campaign_quality))); cpc=np.clip(.45*s.bid*(1+.05*np.log1p(competition)),.05,None)
    if stochastic:
        imp_obs=rng.poisson(np.maximum(imp,0)).astype(np.float32); clk=rng.binomial(np.maximum(imp_obs.astype(int),0),np.clip(ctr,0,1)).astype(np.float32); conv=rng.binomial(np.maximum(clk.astype(int),0),np.clip(cvr,0,1)).astype(np.float32)
    else: imp_obs=imp.astype(np.float32); clk=imp_obs*ctr; conv=clk*cvr
    raw=clk*cpc; rem=np.maximum(s.budget-s.spend,0); scale=np.minimum(1,rem/np.maximum(raw,1e-8)); clk*=scale; conv*=scale; raw*=scale; imp_obs*=scale
    s.period_spend=raw.astype(np.float32); s.spend+=s.period_spend; s.impressions=imp_obs; s.clicks=clk; s.conversions=conv; s.revenue=(conv*campaign_value).astype(np.float32); s.cum_revenue+=s.revenue
    shock=rng.normal(0,.025,Q).astype(np.float32) if stochastic else 0.; s.demand=np.maximum(1,s.demand*(1+shock)); s.t+=1
    assert np.all(s.spend<=s.budget+1e-4)
    return s
s0=initial_state(); zero={"bid_delta":np.zeros(C),"budget_delta":np.zeros(C)}; s1=market_step(s0,zero); print("spend",s1.spend.sum(),"revenue",s1.revenue.sum())

## 4. Graph state tensors and structural intervention encoding

Campaign/query state evolves over time. Actions are injected only at campaign nodes so the model must propagate intervention effects through graph structure.

In [ ]:
DYN_DIM=10
campaign_gids=torch.tensor([node_maps["campaign"][c] for c in campaign_ids],dtype=torch.long); query_gids=torch.tensor([node_maps["query"][str(q)] for q in query_ids],dtype=torch.long)
def state_to_node_features(state,action=None):
    x=np.zeros((N,DYN_DIM),np.float32)
    for cid,i in cid_to_i.items():
        g=node_maps["campaign"][cid]; b=state.budget[i]; sp=state.spend[i]; imp=state.impressions[i]; clk=state.clicks[i]; conv=state.conversions[i]
        cum_rev=state.cum_revenue[i]
        x[g]=[np.log1p(b),np.log1p(sp),np.log1p(state.bid[i]),np.log1p(imp),safe_ratio(clk,imp),safe_ratio(conv,clk),safe_ratio(cum_rev,sp),max(0,1-sp/max(b,1e-6)),float(state.t%7)/6,campaign_quality[i]]
    for q,j in qid_to_j.items(): g=node_maps["query"][str(q)]; x[g,0]=np.log1p(state.demand[j]); x[g,8]=float(state.t%7)/6
    for r in prod_tbl.itertuples(): x[node_maps["product"][str(r.product_id)],9]=1
    for g in node_maps["advertiser"].values(): x[g,9]=.5
    for g in node_maps["market"].values(): x[g,9]=.25
    a=np.zeros((N,2),np.float32)
    if action is not None:
        for cid,i in cid_to_i.items(): g=node_maps["campaign"][cid]; a[g]=[action["bid_delta"][i],action["budget_delta"][i]]
    return torch.tensor(x),torch.tensor(a)

## 5. Generate logged trajectories

In [ ]:
def behavior_action(state,scale=.08):
    roas=safe_ratio(state.cum_revenue,np.maximum(state.spend,1e-4)); center=np.tanh((roas-np.median(roas+1e-6))/2)
    return {"bid_delta":np.clip(.03*center+rng.normal(0,scale,C),-.25,.25).astype(np.float32),
            "budget_delta":np.clip(.02*center+rng.normal(0,scale*.65,C),-.18,.18).astype(np.float32)}
def generate_transitions(episodes=22,horizon=TIME_STEPS):
    out=[]
    for ep in tqdm(range(episodes),desc="Simulating trajectories"):
        s=initial_state()
        for t in range(horizon):
            a=behavior_action(s); nxt=market_step(s,a,True); x,at=state_to_node_features(s,a); y,_=state_to_node_features(nxt,None); out.append((x.float(),at.float(),y.float(),ep,t)); s=nxt
    return out
transitions=generate_transitions(); print("transitions",len(transitions))

## 6. Custom relation-aware Graph Transformer world model

No GAT/TransformerConv shortcut. Attention contains relation bias, source/destination node-type bias, intervention bias and learned relation gates. The transition head predicts mean and log-variance for uncertainty-aware rollouts.

In [ ]:
def segment_softmax(scores,dst,n):
    H=scores.shape[1]; mx=torch.full((n,H),-1e9,device=scores.device); mx.scatter_reduce_(0,dst[:,None].expand(-1,H),scores,reduce="amax",include_self=True)
    z=torch.exp(scores-mx[dst]); den=torch.zeros((n,H),device=scores.device); den.index_add_(0,dst,z); return z/(den[dst]+1e-9)
class EdgeAwareRelationGTLayer(nn.Module):
    def __init__(self,d,heads,n_rel,n_types,dropout=.08):
        super().__init__(); assert d%heads==0; self.d=d; self.h=heads; self.dk=d//heads
        self.q=nn.Linear(d,d,bias=False); self.k=nn.Linear(d,d,bias=False); self.v=nn.Linear(d,d,bias=False)
        self.rel_bias=nn.Embedding(n_rel,heads); self.rel_gate=nn.Embedding(n_rel,heads); self.src_type=nn.Embedding(n_types,heads); self.dst_type=nn.Embedding(n_types,heads); self.action_bias=nn.Linear(2,heads,bias=False)
        self.out=nn.Linear(d,d); self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(dropout),nn.Linear(4*d,d)); self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d); self.drop=nn.Dropout(dropout)
    def forward(self,h,edge_index,edge_type,node_type,action):
        src,dst=edge_index; Q=self.q(h).view(-1,self.h,self.dk); K=self.k(h).view(-1,self.h,self.dk); V=self.v(h).view(-1,self.h,self.dk)
        score=(Q[dst]*K[src]).sum(-1)/math.sqrt(self.dk)+self.rel_bias(edge_type)+self.src_type(node_type[src])+self.dst_type(node_type[dst])+self.action_bias(action[src])
        alpha=segment_softmax(score,dst,h.shape[0]); gate=torch.sigmoid(self.rel_gate(edge_type)); msg=(alpha*gate)[...,None]*V[src]
        agg=torch.zeros((h.shape[0],self.h,self.dk),device=h.device); agg.index_add_(0,dst,msg); z=self.n1(h+self.drop(self.out(agg.reshape(h.shape[0],-1)))); return self.n2(z+self.drop(self.ff(z))),alpha,gate
class GraphWorldModel(nn.Module):
    def __init__(self,in_dim,d=96,heads=4,layers=3):
        super().__init__(); self.inp=nn.Linear(in_dim,d); self.type_emb=nn.Embedding(len(NODE_TYPES),d); self.action_in=nn.Linear(2,d,bias=False)
        self.layers=nn.ModuleList([EdgeAwareRelationGTLayer(d,heads,num_relations,len(NODE_TYPES)) for _ in range(layers)])
        self.mu=nn.Linear(d,in_dim); self.logvar=nn.Linear(d,in_dim); self.reconstruct=nn.Linear(d,in_dim)
    def forward(self,x,action,edge_index,edge_type,node_type):
        h=self.inp(x)+self.type_emb(node_type)+self.action_in(action); att=[]
        for layer in self.layers: h,a,g=layer(h,edge_index,edge_type,node_type,action); att.append((a,g))
        return {"mu":self.mu(h),"logvar":torch.clamp(self.logvar(h),-6,4),"h":h,"rec":self.reconstruct(h),"attn":att}

In [ ]:
eps=sorted({z[3] for z in transitions}); ntr=max(1,int(.7*len(eps))); nva=max(1,int(.15*len(eps)))
train_eps=set(eps[:ntr]); val_eps=set(eps[ntr:ntr+nva]); test_eps=set(eps[ntr+nva:])
train_data=[z for z in transitions if z[3] in train_eps]; val_data=[z for z in transitions if z[3] in val_eps]; test_data=[z for z in transitions if z[3] in test_eps]
assert train_eps.isdisjoint(val_eps|test_eps) and val_eps.isdisjoint(test_eps)
X=torch.stack([z[0] for z in train_data]); feat_mean=X.mean((0,1)); feat_std=X.std((0,1)).clamp_min(1e-4)
def norm_x(x): return (x-feat_mean)/feat_std
def denorm_x(x): return x*feat_std+feat_mean
edge_index_d=edge_index.to(DEVICE); edge_type_d=edge_type.to(DEVICE); node_type_d=torch.tensor(node_type,dtype=torch.long,device=DEVICE)
model=GraphWorldModel(DYN_DIM).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
dynamic_mask=torch.zeros((N,DYN_DIM),dtype=torch.bool); dynamic_mask[campaign_gids,:9]=True; dynamic_mask[query_gids,0]=True; dynamic_mask=dynamic_mask.to(DEVICE)
def gnll(mu,lv,y,m): return (.5*(lv+(y-mu)**2*torch.exp(-lv)))[m].mean()
def run_epoch(data,train=True):
    model.train(train); total=0
    for idx in (np.random.permutation(len(data)) if train else np.arange(len(data))):
        x,a,y,ep,t=data[idx]; x=norm_x(x).to(DEVICE); y=norm_x(y).to(DEVICE); a=a.to(DEVICE); out=model(x,a,edge_index_d,edge_type_d,node_type_d); nll=gnll(out["mu"],out["logvar"],y,dynamic_mask)
        m=(torch.rand_like(x)<.08)&dynamic_mask; xm=x.clone(); xm[m]=0; outm=model(xm,a,edge_index_d,edge_type_d,node_type_d); rec=F.smooth_l1_loss(outm["rec"][m],x[m]) if m.any() else torch.tensor(0.,device=DEVICE)
        src,dst=edge_index_d; smooth=(out["h"][src]-out["h"][dst]).pow(2).mean(); loss=nll+.12*rec+.002*smooth
        if train: opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1); opt.step()
        total+=float(loss.detach())
    return total/max(len(data),1)
best=1e9; best_sd=None; pat=0
for epoch in range(20):
    tr=run_epoch(train_data,True); va=run_epoch(val_data,False); print(epoch,tr,va)
    if va<best-1e-4: best=va; pat=0; best_sd={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else: pat+=1
    if pat>=4: break
model.load_state_dict(best_sd); print("best val",best)

## 7. World-model evaluation

In [ ]:
@torch.no_grad()
def predict_next(x,a):
    out=model(norm_x(x).to(DEVICE),a.to(DEVICE),edge_index_d,edge_type_d,node_type_d); return denorm_x(out["mu"].cpu()), torch.exp(.5*out["logvar"].cpu())*feat_std, out
errs=[]; zs=[]
for x,a,y,ep,t in test_data:
    mu,sig,_=predict_next(x,a); diff=(mu-y)[dynamic_mask.cpu()]; errs.append(float(torch.sqrt((diff**2).mean()))); zs.append(float((diff.abs()/(sig[dynamic_mask.cpu()]+1e-5)).mean()))
print("test dynamic RMSE",np.mean(errs),"mean standardized abs residual",np.mean(zs))

## 8. Custom log-domain Sinkhorn OT for incremental budget transport

In [ ]:
def log_sinkhorn(logK,log_a,log_b,iters=80):
    u=torch.zeros_like(log_a); v=torch.zeros_like(log_b)
    for _ in range(iters): u=log_a-torch.logsumexp(logK+v[None,:],1); v=log_b-torch.logsumexp(logK+u[:,None],0)
    return torch.exp(logK+u[:,None]+v[None,:])
def budget_transport(opportunity,uncertainty,saturation,total_budget=10000.,eps=.15):
    opp=torch.as_tensor(opportunity,dtype=torch.float32); unc=torch.as_tensor(uncertainty,dtype=torch.float32); sat=torch.as_tensor(saturation,dtype=torch.float32); n=len(opp); m=min(8,max(2,n//20))
    a=torch.full((m,),1/m); b=torch.softmax(opp-.7*unc-.35*sat,0); cost=(-opp+.7*unc+.35*sat)[None,:].repeat(m,1); T=log_sinkhorn(-cost/eps,torch.log(a),torch.log(b)); return (T.sum(0)*total_budget).numpy(),T.numpy()

## 9. Uncertainty-aware Graph-MPC

Generate candidate action sequences, roll the GTN world model forward, penalize uncertainty, execute only the first action, then re-plan.

In [ ]:
def random_candidate(scale=.09): return {"bid_delta":np.clip(rng.normal(0,scale,C),-.2,.2).astype(np.float32),"budget_delta":np.clip(rng.normal(0,scale*.7,C),-.15,.15).astype(np.float32)}
def campaign_reward(x):
    c=x[campaign_gids]; spend=torch.expm1(c[:,1]).clamp_min(0); return (.45*c[:,6].clamp(0,20)+.25*c[:,5].clamp(0,1)+.1*c[:,7].clamp(0,1)-.00002*spend).mean()
@torch.no_grad()
def world_rollout(x0,seq):
    # Legacy Gaussian-GTN ablation: only supervised dynamic coordinates are rolled forward;
    # static/product/brand/market coordinates are frozen rather than replaced by unconstrained outputs.
    x=x0.clone(); total=torch.tensor(0.); risk=torch.tensor(0.); states=[]; dm=dynamic_mask.cpu()
    for act in seq:
        _,a=state_to_node_features(initial_state(),act); out=model(norm_x(x).to(DEVICE),a.to(DEVICE),edge_index_d,edge_type_d,node_type_d); mu=denorm_x(out["mu"].cpu()); sig=torch.exp(.5*out["logvar"].cpu())*feat_std; nx=x.clone(); nx[dm]=mu[dm]; x=nx; total+=campaign_reward(x); risk+=sig[dm].mean(); states.append((x.clone(),sig.clone()))
    return total,risk,states
def graph_mpc(x_now,horizon=3,candidates=48,risk_lambda=.10):
    best=None
    for _ in range(candidates):
        seq=[random_candidate() for _ in range(horizon)]; rew,risk,states=world_rollout(x_now,seq); score=float(rew-risk_lambda*risk)
        if best is None or score>best["score"]: best={"score":score,"sequence":seq,"reward":float(rew),"risk":float(risk),"states":states}
    return best

## 10. Policy comparison in the controlled simulator

In [ ]:
def ot_action(s):
    roas=safe_ratio(s.cum_revenue,np.maximum(s.spend,1e-4)); sat=np.clip(safe_ratio(s.spend,s.budget),0,1); alloc,_=budget_transport(roas,np.full(C,.15,np.float32),sat,float(np.maximum(s.budget-s.spend,0).sum()*.03)); centered=alloc/max(alloc.mean(),1e-6)-1
    return {"bid_delta":np.clip(.025*np.tanh(roas-np.median(roas)),-.08,.08).astype(np.float32),"budget_delta":np.clip(centered*.03,-.1,.1).astype(np.float32)}
def evaluate_policy(policy,episodes=3,horizon=8):
    vals=[]; violations=[]
    for _ in range(episodes):
        s=initial_state(); rev=0.; spend=0.; vio=0
        for t in range(horizon):
            if policy=="zero": act={"bid_delta":np.zeros(C,np.float32),"budget_delta":np.zeros(C,np.float32)}
            elif policy=="random": act=behavior_action(s,.05)
            elif policy=="ot": act=ot_action(s)
            else:
                x,_=state_to_node_features(s,None); act=graph_mpc(x,horizon=2,candidates=24,risk_lambda=.08)["sequence"][0]
            s=market_step(s,act,True); rev+=float(s.revenue.sum()); spend+=float(s.period_spend.sum()); vio+=int(np.any(s.spend>s.budget+1e-4))
        vals.append(rev/max(spend,1e-6)); violations.append(vio)
    return {"policy":policy,"simulated_roas":float(np.mean(vals)),"std":float(np.std(vals)),"violations":int(np.sum(violations))}
pd.DataFrame([evaluate_policy(p) for p in ["zero","random","ot","mpc"]])

## 11. Safety gate: EXECUTE / REVISE / ABSTAIN

In [ ]:
def safety_gate(x_now,action,uncertainty_threshold=1.25):
    if np.max(np.abs(action["bid_delta"]))>.25: return "REVISE","bid-step limit"
    if np.max(np.abs(action["budget_delta"]))>.18: return "REVISE","budget-step limit"
    rew,risk,states=world_rollout(x_now,[action]); nx,sig=states[-1]
    if not torch.isfinite(nx).all(): return "ABSTAIN","non-finite state"
    if float(sig[campaign_gids].mean())>uncertainty_threshold: return "ABSTAIN","out-of-support uncertainty"
    return "EXECUTE",f"projected_reward={float(rew):.4f}"
s=initial_state(); x,_=state_to_node_features(s,None); plan=graph_mpc(x,horizon=2,candidates=20); print(safety_gate(x,plan["sequence"][0]))

## 12. Agent-facing tool facade

Plain Python tools can later be exposed through MCP/function calling. This notebook does not call private Amazon Ads APIs.

In [ ]:
class GraphWorldAdsTools:
    def __init__(self,state): self.state=state
    def inspect_market_graph(self): return {"nodes":N,"edges":int(edge_index.shape[1]),"campaigns":C,"queries":Q,"markets":market_tbl.locale.tolist()}
    def forecast_campaign_state(self,action):
        x,_=state_to_node_features(self.state,None); r,u,_=world_rollout(x,[action]); return {"reward_proxy":float(r),"uncertainty":float(u)}
    def optimize_budget_transport(self): return ot_action(self.state)
    def search_action_sequence(self,horizon=3):
        x,_=state_to_node_features(self.state,None); return graph_mpc(x,horizon=horizon,candidates=36)
    def validate_action(self,action):
        x,_=state_to_node_features(self.state,None); return safety_gate(x,action)
tools=GraphWorldAdsTools(initial_state()); print(tools.inspect_market_graph())

## 13. Research evaluation / ablations

### World-model quality
- one-step RMSE / Gaussian NLL;
- uncertainty calibration;
- multi-step rollout drift;
- cross-locale generalization (e.g. train US+ES, test JP);
- out-of-distribution intervention tests.

### Planner quality
- cumulative simulated advertiser value;
- simulated ROAS and budget utilization;
- constraint violations;
- regret vs a simulator oracle on a reduced action grid;
- action efficiency and abstention quality.

### Ablations
1. node-wise MLP world model;
2. GraphSAGE world model;
3. relation-agnostic Graph Transformer;
4. full relation-aware GraphWorld model;
5. + OT allocator;
6. + Graph-MPC;
7. + uncertainty gate.

### Claim discipline
Defensible claim: **real Amazon query/product/brand/locale structure + controlled semi-synthetic advertising dynamics**. Do not present simulator bids/clicks/conversions as Amazon production outcomes.

## 14. High-value extensions

- explicit locale-expansion intervention nodes;
- auction-equilibrium regularizer;
- distributional transition heads (mixture/quantile);
- latent competitor-policy embeddings;
- conservative offline planning / behavior-distance constraints;
- graph-certified constraint verifier;
- MCP wrapper for inspect/simulate/optimize/validate/execute;
- CUDA/Triton kernel for custom relation-aware attention after modeling stabilizes.

# GraphWorld-Ads-X — Audited Research Upgrade
## Intervention-Aware Multi-Agent Generative Graph World Model

This section is **additive**: every original GraphWorld-Ads cell above remains unchanged and serves as a baseline.

The X-model addresses the main scientific limitations of the baseline:

1. **Semantic Amazon graph** — multilingual query/product/brand representations are injected into graph attention.
2. **Structural interventions** — actions become explicit action nodes and can add, suppress, or reweight campaign–query / campaign–market edges.
3. **Typed state transitions** — only valid dynamic state is predicted; immutable product/brand/market semantics never drift.
4. **Generative graph flow matching** — the world model produces multiple plausible futures rather than a single Gaussian average.
5. **True opponent modeling** — non-focal advertisers act using independently learned latent policies.
6. **Relevance-adjusted GSP/CPC auction + budget pacing** — a stronger controlled marketplace than softmax share.
7. **Structured cross-market/channel optimal transport** — transport sources and destinations have non-identical geometry.
8. **Risk-sensitive Graph-MPPI** — hierarchical intervention search over focal campaigns with CVaR-aware trajectory scoring.
9. **Ensemble + OOD uncertainty** — epistemic disagreement, trajectory dispersion, and support distance.
10. **Offline policy evaluation** — SNIPS and doubly robust estimates on held-out logged interventions.
11. **US+ES → JP cold start** — Japan is withheld from transition supervision to test structural/semantic transfer.
12. **Planner/verifier tools** — inspect → propose → simulate → verify → execute/abstain boundary suitable for an MCP wrapper.

**Claims discipline:** ESCI graph structure and text are public Amazon shopping data. Campaigns, auction outcomes, budgets, interventions, and counterfactual futures remain controlled/semi-synthetic research constructs.

In [ ]:
# X-model dependencies and controls
!pip -q install "sentence-transformers>=3.0.0" "scikit-learn>=1.4.0"
from dataclasses import dataclass,field
from typing import Dict,List,Tuple,Optional
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import mean_squared_error
X_SEED=SEED+101; x_rng=np.random.default_rng(X_SEED); torch.manual_seed(X_SEED)
X_SEM_DIM=64; X_DMODEL=112; X_HEADS=4; X_LAYERS=3
if FULL_RESEARCH_RUN:
    X_EPISODES=16; X_HORIZON=14; X_FLOW_STEPS=8; X_ENSEMBLE_SIZE=2; X_TRAIN_EPOCHS=12; X_FAST_DEMO=False
else:
    X_EPISODES=8; X_HORIZON=8; X_FLOW_STEPS=6; X_ENSEMBLE_SIZE=2; X_TRAIN_EPOCHS=7; X_FAST_DEMO=True
X_QUERY_CLUSTERS=min(32,max(4,int(np.sqrt(max(Q,1))))); ORIGINAL_BASELINE_CELLS=33
print({"semantic_dim":X_SEM_DIM,"ensemble":X_ENSEMBLE_SIZE,"query_clusters":X_QUERY_CLUSTERS,"device":str(DEVICE),"full_research_run":FULL_RESEARCH_RUN})

## X1. Semantic Amazon graph

The original graph used ESCI mainly as topology/relevance. Here, query text, product titles, brands, campaign descriptors, and market names become node features.

Primary encoder: `multilingual-e5-small` (appropriate for US/ES/JP text).  
Fallback: deterministic hashing + truncated SVD if the model cannot be downloaded.

In [ ]:
def _fit_semantic_encoder(texts, dim=X_SEM_DIM):
    texts=[str(t) if str(t).strip() else "[EMPTY]" for t in texts]
    try:
        from sentence_transformers import SentenceTransformer
        sem_model=SentenceTransformer("intfloat/multilingual-e5-small",device=str(DEVICE))
        emb=sem_model.encode(texts,batch_size=128,normalize_embeddings=True,show_progress_bar=True)
        emb=np.asarray(emb,np.float32)
        if emb.shape[1]>dim:
            svd=TruncatedSVD(n_components=dim,random_state=X_SEED); emb=svd.fit_transform(emb).astype(np.float32)
        elif emb.shape[1]<dim: emb=np.pad(emb,((0,0),(0,dim-emb.shape[1])))
    except Exception as e:
        print("E5 unavailable; deterministic hashing/SVD fallback:",repr(e))
        hv=HashingVectorizer(n_features=2048,alternate_sign=False,norm="l2",analyzer="char_wb",ngram_range=(3,5)); H=hv.transform(texts)
        ncomp=min(dim,max(2,min(H.shape[0]-1,H.shape[1]-1))); svd=TruncatedSVD(n_components=ncomp,random_state=X_SEED); emb=svd.fit_transform(H).astype(np.float32)
        if emb.shape[1]<dim: emb=np.pad(emb,((0,0),(0,dim-emb.shape[1])))
    emb=emb.astype(np.float32); emb/=np.maximum(np.linalg.norm(emb,axis=1,keepdims=True),1e-8)
    return emb,{"backend":"multilingual-e5-small" if 'sem_model' in locals() else "hashing+svd"}

node_text=["[EMPTY]"]*N
for r in advertiser_tbl.itertuples(): node_text[node_maps["advertiser"][str(r.advertiser_id)]]=f"passage: advertiser brand {r.advertiser_id}"
for r in campaign_seed.itertuples(): node_text[node_maps["campaign"][r.campaign_id]]=f"passage: campaign brand {r.brand}; marketplace {r.locale}"
for r in query_tbl.itertuples(): node_text[node_maps["query"][str(r.query_id)]]=f"query: {r.query}; marketplace {r.locale}"
for r in prod_tbl.itertuples(): node_text[node_maps["product"][str(r.product_id)]]=f"passage: {r.product_text}; title {r.title}; brand {r.brand}; marketplace {r.locale}"
for r in market_tbl.itertuples(): node_text[node_maps["market"][str(r.locale)]]=f"passage: marketplace locale {r.locale}"
node_sem_np,semantic_meta=_fit_semantic_encoder(node_text); node_sem=torch.tensor(node_sem_np,dtype=torch.float32)
assert node_sem.shape==(N,X_SEM_DIM) and np.isfinite(node_sem_np).all()
print("semantic backend:",semantic_meta,"shape:",node_sem.shape)

In [ ]:
# Query semantic clusters are later used as OT destinations and cross-market opportunity units.
q_sem = node_sem_np[query_gids.numpy()]
if len(q_sem) >= X_QUERY_CLUSTERS:
    qcluster_model = MiniBatchKMeans(n_clusters=X_QUERY_CLUSTERS, random_state=X_SEED, batch_size=1024, n_init="auto")
    query_cluster = qcluster_model.fit_predict(q_sem)
else:
    query_cluster = np.arange(len(q_sem)) % max(1, len(q_sem))

query_cluster = np.asarray(query_cluster, np.int64)
qid_to_cluster = {qid:int(query_cluster[j]) for qid,j in qid_to_j.items()}
print("query clusters:", len(np.unique(query_cluster)))

## X2. Structural intervention graph

Actions are no longer just two floats attached to campaign nodes.

Each decision creates an **Action Node** per touched campaign. Intervention relations include:
- bid/budget action → campaign;
- add keyword → campaign–query edge;
- suppress keyword → campaign–query edge attenuation;
- expand market → campaign–market edge.

This allows an intervention to change graph topology as well as node state.

In [ ]:
X_NODE_TYPES=dict(NODE_TYPES); X_NODE_TYPES["action"]=len(X_NODE_TYPES)
X_BASE_REL=num_relations
_names=["action_campaign","campaign_action","add_query","query_added_by","drop_query","query_dropped_by","expand_market","market_expanded_by","target_query","query_targeted_by","access_market","market_accessed_by"]
for _i,_n in enumerate(_names): globals()["X_REL_"+_n.upper()]=X_BASE_REL+_i
X_NUM_RELATIONS=X_BASE_REL+len(_names)

@dataclass
class Intervention:
    bid_delta:np.ndarray; budget_delta:np.ndarray
    add_query_edges:List[Tuple[int,int]]=field(default_factory=list)
    drop_query_edges:List[Tuple[int,int]]=field(default_factory=list)
    expand_market:List[Tuple[int,str]]=field(default_factory=list)
    def copy(self): return Intervention(self.bid_delta.copy(),self.budget_delta.copy(),list(self.add_query_edges),list(self.drop_query_edges),list(self.expand_market))

def intervention_from_legacy(action): return Intervention(np.asarray(action["bid_delta"],np.float32).copy(),np.asarray(action["budget_delta"],np.float32).copy())

_qp_rel={}
for rr in es.itertuples():
    qg=node_maps["query"].get(str(rr.query_id)); pg=node_maps["product"].get(str(rr.product_id))
    if qg is not None and pg is not None:
        _qp_rel[(qg,pg)]=max(_qp_rel.get((qg,pg),0.),float(rr.rel)); _qp_rel[(pg,qg)]=_qp_rel[(qg,pg)]
_gid_to_ci={node_maps["campaign"][c]:i for i,c in enumerate(campaign_ids)}; _gid_to_qj={node_maps["query"][str(q)]:j for j,q in enumerate(query_ids)}
x_base_edge_weight=torch.ones(edge_type.shape[0],dtype=torch.float32)
for kk,(ss,dd,tt) in enumerate(zip(edge_index[0].tolist(),edge_index[1].tolist(),edge_type.tolist())):
    if (ss,dd) in _qp_rel: x_base_edge_weight[kk]=max(.05,float(_qp_rel[(ss,dd)]))
    elif ss in _gid_to_ci and dd in _gid_to_qj: x_base_edge_weight[kk]=max(.05,float(rel_cq[_gid_to_ci[ss],_gid_to_qj[dd]]))
    elif dd in _gid_to_ci and ss in _gid_to_qj: x_base_edge_weight[kk]=max(.05,float(rel_cq[_gid_to_ci[dd],_gid_to_qj[ss]]))

def _structure_arrays(structure=None):
    if structure is None: return base_eligibility, initial_market_access
    if hasattr(structure,"eligibility"): return structure.eligibility,structure.market_access
    if isinstance(structure,dict): return structure["eligibility"],structure["market_access"]
    return structure

def build_intervention_graph(iv:Intervention,structure=None):
    elig,access=_structure_arrays(structure)
    touched=sorted(set(np.where((np.abs(iv.bid_delta)+np.abs(iv.budget_delta))>1e-8)[0].tolist()+[c for c,q in iv.add_query_edges]+[c for c,q in iv.drop_query_edges]+[c for c,m in iv.expand_market]))
    action_gid={ci:N+k for k,ci in enumerate(touched)}; n_ext=N+len(touched)
    ei=edge_index.clone(); et=edge_type.clone(); ew=x_base_edge_weight.clone()
    S=[];D=[];T=[];W=[]
    # Persistent current targeting and market-access state.
    for ci,qj in np.argwhere(elig>0):
        cg=node_maps["campaign"][campaign_ids[int(ci)]]; qg=node_maps["query"][str(query_ids[int(qj)])]
        w=max(.05,float(effective_rel_cq[int(ci),int(qj)])); S += [cg,qg]; D += [qg,cg]; T += [X_REL_TARGET_QUERY,X_REL_QUERY_TARGETED_BY]; W += [w,w]
    for ci,mk in np.argwhere(access>0):
        cg=node_maps["campaign"][campaign_ids[int(ci)]]; mg=node_maps["market"][MARKETS[int(mk)]]
        S += [cg,mg]; D += [mg,cg]; T += [X_REL_ACCESS_MARKET,X_REL_MARKET_ACCESSED_BY]; W += [1.,1.]
    for ci in touched:
        ag=action_gid[ci]; cg=node_maps["campaign"][campaign_ids[ci]]; mag=float(np.clip(abs(iv.bid_delta[ci])+abs(iv.budget_delta[ci]),.05,.5))
        S += [ag,cg]; D += [cg,ag]; T += [X_REL_ACTION_CAMPAIGN,X_REL_CAMPAIGN_ACTION]; W += [mag,mag]
    for ci,qj in iv.add_query_edges:
        if 0<=ci<C and 0<=qj<Q:
            cg=node_maps["campaign"][campaign_ids[ci]]; qg=node_maps["query"][str(query_ids[qj])]; w=max(.05,float(effective_rel_cq[ci,qj]))
            S += [cg,qg]; D += [qg,cg]; T += [X_REL_ADD_QUERY,X_REL_QUERY_ADDED_BY]; W += [w,w]
    for ci,qj in iv.drop_query_edges:
        if 0<=ci<C and 0<=qj<Q:
            cg=node_maps["campaign"][campaign_ids[ci]]; qg=node_maps["query"][str(query_ids[qj])]
            S += [cg,qg]; D += [qg,cg]; T += [X_REL_DROP_QUERY,X_REL_QUERY_DROPPED_BY]; W += [.2,.2]
    for ci,loc in iv.expand_market:
        if 0<=ci<C and str(loc) in node_maps["market"]:
            cg=node_maps["campaign"][campaign_ids[ci]]; mg=node_maps["market"][str(loc)]
            S += [cg,mg]; D += [mg,cg]; T += [X_REL_EXPAND_MARKET,X_REL_MARKET_EXPANDED_BY]; W += [1.,1.]
    if S:
        ei=torch.cat([ei,torch.tensor([S,D],dtype=torch.long)],1); et=torch.cat([et,torch.tensor(T,dtype=torch.long)]); ew=torch.cat([ew,torch.tensor(W,dtype=torch.float32)])
    nt=torch.tensor(np.concatenate([node_type,np.full(len(touched),X_NODE_TYPES["action"],np.int64)]),dtype=torch.long)
    sem=torch.cat([node_sem,torch.zeros((len(touched),X_SEM_DIM))],0); act=torch.zeros((n_ext,4),dtype=torch.float32)
    for ci,ag in action_gid.items():
        act[ag]=torch.tensor([float(iv.bid_delta[ci]),float(iv.budget_delta[ci]),float(sum(c==ci for c,q in iv.add_query_edges)-sum(c==ci for c,q in iv.drop_query_edges)),float(sum(c==ci for c,m in iv.expand_market))])
    return {"edge_index":ei,"edge_type":et,"edge_weight":ew,"node_type":nt,"semantic":sem,"action":act,"n_ext":n_ext,"action_gid":action_gid}

## X3. Relevance-adjusted GSP/CPC auction, pacing, and structural eligibility

The original simulator remains a baseline. This controlled environment adds:
- top-slot ranking by relevance × quality × bid;
- GSP-style next-score pricing;
- click/conversion generation by slot and relevance;
- daily budget pacing;
- intervention-controlled query eligibility;
- cross-market expansion state.

In [ ]:
MARKETS=market_tbl.locale.astype(str).tolist(); market_to_k={m:k for k,m in enumerate(MARKETS)}
campaign_locale=campaign_seed.locale.astype(str).map(market_to_k).to_numpy(); query_locale=query_tbl.locale.astype(str).map(market_to_k).to_numpy()

# Cross-market semantic opportunity matrix. Real ESCI relevance dominates where observed;
# foreign-market opportunities are semi-synthetic semantic candidates, explicitly separated from real labels.
q_sem=node_sem_np[query_gids.numpy()]
camp_sem=np.zeros((C,X_SEM_DIM),np.float32)
for ci,cid in enumerate(campaign_ids):
    brand=campaign_seed.loc[ci,"brand"]; loc=campaign_seed.loc[ci,"locale"]
    pids=prod_tbl[(prod_tbl.brand==brand)&(prod_tbl.locale==loc)].product_id.astype(str).tolist()
    pg=[node_maps["product"][p] for p in pids if p in node_maps["product"]]
    camp_sem[ci]=node_sem_np[pg].mean(0) if pg else node_sem_np[node_maps["campaign"][cid]]
    camp_sem[ci]/=max(np.linalg.norm(camp_sem[ci]),1e-8)
semantic_cq=np.clip((camp_sem@q_sem.T+1.)/2.,0,1).astype(np.float32)
effective_rel_cq=rel_cq.copy(); targetable_cq=(rel_cq>0)
for ci in range(C):
    for mk in range(len(MARKETS)):
        qs=np.where(query_locale==mk)[0]
        if not len(qs): continue
        top=qs[np.argsort(semantic_cq[ci,qs])[-min(12,len(qs)):]]
        # semantic affinity is a simulator proxy, not an ESCI relevance label
        effective_rel_cq[ci,top]=np.maximum(effective_rel_cq[ci,top],.42*semantic_cq[ci,top])
        targetable_cq[ci,top]=True

# Campaign competition affinity from overlap in targetable query opportunity.
_tc=targetable_cq.astype(np.float32); _norm=np.sqrt((_tc*_tc).sum(1,keepdims=True)).clip(1e-6); campaign_competition=(_tc@_tc.T)/(_norm@_norm.T); np.fill_diagonal(campaign_competition,0.)

# Start with only a subset of real supported queries active; this makes add-query interventions non-trivial.
base_eligibility=np.zeros((C,Q),np.float32)
for ci in range(C):
    real=np.where(rel_cq[ci]>0)[0]
    if len(real):
        order=real[np.argsort(rel_cq[ci,real])[::-1]]; k=max(1,int(np.ceil(.65*len(order))))
        if len(order)>1: k=min(k,len(order)-1)
        base_eligibility[ci,order[:k]]=1.

initial_market_access=np.zeros((C,len(MARKETS)),np.float32)
for i,k in enumerate(campaign_locale): initial_market_access[i,k]=1.

@dataclass
class XMarketState:
    base:MarketState; eligibility:np.ndarray; market_access:np.ndarray

def initial_x_state(): return XMarketState(initial_state(),base_eligibility.copy(),initial_market_access.copy())
def clone_x_state(xs): return XMarketState(clone_state(xs.base),xs.eligibility.copy(),xs.market_access.copy())
def apply_structural_intervention(xs,iv):
    out=clone_x_state(xs)
    for ci,qj in iv.add_query_edges:
        if 0<=ci<C and 0<=qj<Q and targetable_cq[ci,qj]>0: out.eligibility[ci,qj]=1.
    for ci,qj in iv.drop_query_edges:
        if 0<=ci<C and 0<=qj<Q: out.eligibility[ci,qj]=0.
    for ci,loc in iv.expand_market:
        if 0<=ci<C and str(loc) in market_to_k: out.market_access[ci,market_to_k[str(loc)]]=1.
    return out

def market_step_gsp(xs,iv,stochastic=True,slots=(1.0,.62,.38),reserve=.02):
    out=apply_structural_intervention(xs,iv); s=out.base
    s.bid=np.clip(s.bid*np.exp(iv.bid_delta),.12,8.)
    proposed=np.maximum(50.,s.budget*(1.+iv.budget_delta)); s.budget=np.maximum(proposed,s.spend)
    access_q=out.market_access[:,query_locale]; eligible=(out.eligibility>0)&(access_q>0)
    quality=(campaign_quality[:,None]*np.maximum(effective_rel_cq,.02))*eligible; score=s.bid[:,None]*quality
    if C==0: return out
    K=min(len(slots)+1,C); idx=np.argpartition(score,-K,axis=0)[-K:,:]; vals=np.take_along_axis(score,idx,0); order=np.argsort(vals,axis=0)[::-1]; idx=np.take_along_axis(idx,order,0); vals=np.take_along_axis(vals,order,0)
    demand=np.maximum(1.,s.demand*(1.+.10*np.sin(2*np.pi*(s.t%7)/7)))
    imp=np.zeros(C,np.float32); clicks=np.zeros(C,np.float32); conv=np.zeros(C,np.float32); spend_inc=np.zeros(C,np.float32)
    remaining=np.maximum(s.budget-s.spend,0); periods_left=max(1,X_HORIZON-(s.t%X_HORIZON)); pace_cap=remaining/periods_left
    q=np.arange(Q)
    for slot,mult in enumerate(slots):
        if slot>=K: break
        ci=idx[slot]; sc=vals[slot]; next_sc=vals[slot+1] if slot+1<K else np.full(Q,reserve,np.float32)
        valid=(sc>0)&eligible[ci,q]
        if not np.any(valid): continue
        imp_q=demand*float(mult)*valid; rel=np.clip(effective_rel_cq[ci,q],0,1)
        ctr=1/(1+np.exp(-(-3.15+2.5*rel+.35*float(mult)+.12*np.log1p(s.bid[ci])))); cvr=1/(1+np.exp(-(-3.75+1.8*rel+.25*campaign_quality[ci])))
        denom=np.maximum(quality[ci,q],1e-5); cpc=np.minimum(s.bid[ci],np.maximum(reserve,next_sc/denom+reserve))
        if stochastic:
            imp_obs=x_rng.poisson(np.maximum(imp_q,0)).astype(np.float32); clk=x_rng.binomial(imp_obs.astype(int),np.clip(ctr,0,1)).astype(np.float32); cv=x_rng.binomial(clk.astype(int),np.clip(cvr,0,1)).astype(np.float32)
        else: imp_obs=imp_q.astype(np.float32); clk=imp_obs*ctr; cv=clk*cvr
        cost=clk*cpc
        for c in np.unique(ci[valid]):
            mask=valid&(ci==c); raw=float(cost[mask].sum()); scale=min(1.,float(pace_cap[c]/max(raw,1e-8)))
            imp[c]+=float(imp_obs[mask].sum())*scale; clicks[c]+=float(clk[mask].sum())*scale; conv[c]+=float(cv[mask].sum())*scale; spend_inc[c]+=raw*scale
    s.period_spend=spend_inc; s.spend+=spend_inc; s.impressions=imp; s.clicks=clicks; s.conversions=conv; s.revenue=conv*campaign_value; s.cum_revenue+=s.revenue
    shock=x_rng.normal(0,.02,Q).astype(np.float32) if stochastic else 0.; s.demand=np.maximum(1.,s.demand*(1.+shock)); s.t+=1; out.base=s
    assert np.all(s.spend<=s.budget+1e-4)
    return out

_xs=initial_x_state(); _iv=Intervention(np.zeros(C,np.float32),np.zeros(C,np.float32)); _nxt=market_step_gsp(_xs,_iv,stochastic=False)
assert np.all(_nxt.base.spend<=_nxt.base.budget+1e-4)
print("GSP sanity spend/revenue",float(_nxt.base.spend.sum()),float(_nxt.base.revenue.sum()),"initial targeting density",float(base_eligibility.mean()))

## X4. Logged multi-agent trajectories with structural interventions

The behavior policy now acts per advertiser/campaign and occasionally changes graph structure. Logged actions are retained explicitly so opponent policies and OPE can be trained from the same trajectories.

In [ ]:
adv_ids=advertiser_tbl.advertiser_id.astype(str).tolist(); adv_to_i={a:i for i,a in enumerate(adv_ids)}
campaign_adv_idx=np.array([adv_to_i[str(a)] for a in campaign_seed.advertiser_id],np.int64); A=len(adv_ids)
_adv_counts=np.bincount(campaign_adv_idx,minlength=A); FOCAL_ADV=int(np.argmax(_adv_counts)) if A else 0; focal_campaigns=np.where(campaign_adv_idx==FOCAL_ADV)[0]
print("focal advertiser",adv_ids[FOCAL_ADV] if A else None,"campaigns",len(focal_campaigns))
def _behavior_means(xs):
    s=xs.base; roas=safe_ratio(s.cum_revenue,np.maximum(s.spend,1e-4)); med=np.median(roas+1e-6); m=.025*np.tanh((roas-med)/2); return m,.5*m
def _sample_bounded(mean,sd,limit):
    z=mean+x_rng.normal(0,sd,len(mean)); return (limit*np.tanh(z/limit)).astype(np.float32)
def behavior_intervention(xs,structural_prob=.15):
    bm,um=_behavior_means(xs)
    # Focal advertiser moves first; competitors react in proportion to shared query opportunity.
    bid=_sample_bounded(bm,.07,.22); bud=_sample_bounded(um,.045,.15)
    if len(focal_campaigns):
        focal_signal=float(bid[focal_campaigns].mean()); affinity=campaign_competition[:,focal_campaigns].mean(1); nonf=np.ones(C,bool); nonf[focal_campaigns]=False
        response=.30*affinity*focal_signal
        bid[nonf]=_sample_bounded((bm+response)[nonf],.07,.22); bud[nonf]=_sample_bounded((um+.12*response)[nonf],.045,.15)
    adds=[]; drops=[]; expands=[]
    if C and Q and x_rng.random()<structural_prob:
        ci=int(x_rng.integers(C)); addcand=np.where(targetable_cq[ci]&(xs.eligibility[ci]==0))[0]; live=np.where(xs.eligibility[ci]>0)[0]
        if len(addcand) and x_rng.random()<.60: adds.append((ci,int(x_rng.choice(addcand))))
        elif len(live): drops.append((ci,int(x_rng.choice(live))))
        if x_rng.random()<.40:
            foreign=[m for m in MARKETS if xs.market_access[ci,market_to_k[m]]==0]
            if foreign:
                loc=str(x_rng.choice(foreign)); expands.append((ci,loc)); qs=np.where((query_locale==market_to_k[loc])&targetable_cq[ci]&(xs.eligibility[ci]==0))[0]
                if len(qs): adds.append((ci,int(qs[np.argmax(effective_rel_cq[ci,qs])])))
    return Intervention(bid,bud,adds,drops,expands)
def x_state_to_features(xs): return state_to_node_features(xs.base,None)[0].float()
def generate_x_trajectories(episodes=X_EPISODES,horizon=X_HORIZON):
    rows=[]
    for ep in tqdm(range(episodes),desc="GSP multi-agent trajectories"):
        xs=initial_x_state()
        for t in range(horizon):
            iv=behavior_intervention(xs); nxt=market_step_gsp(xs,iv,True); x=x_state_to_features(xs); y=x_state_to_features(nxt); rows.append({"x":x,"y":y,"iv":iv,"ep":ep,"t":t,"reward":float(nxt.base.revenue.sum()-nxt.base.period_spend.sum()),"state":xs,"next_state":nxt}); xs=nxt
    return rows
x_transitions=generate_x_trajectories(); x_eps=sorted({r["ep"] for r in x_transitions}); _ntr=max(1,int(.7*len(x_eps))); _nva=max(1,int(.15*len(x_eps))); x_train_eps=set(x_eps[:_ntr]); x_val_eps=set(x_eps[_ntr:_ntr+_nva]); x_test_eps=set(x_eps[_ntr+_nva:]); x_train=[r for r in x_transitions if r["ep"] in x_train_eps]; x_val=[r for r in x_transitions if r["ep"] in x_val_eps]; x_test=[r for r in x_transitions if r["ep"] in x_test_eps]; assert x_train_eps.isdisjoint(x_val_eps|x_test_eps) and x_val_eps.isdisjoint(x_test_eps); print(len(x_train),len(x_val),len(x_test))

## X5. Typed dynamic state and graph flow matching

The X-model **never predicts immutable node features**.

Dynamic target:
- campaign: budget, cumulative spend, bid, impressions, CTR, CVR, ROAS, remaining-budget ratio, time;
- query: demand and time;
- graph topology: intervention edges are provided structurally, not hallucinated as dense adjacency.

Flow matching learns a conditional velocity field from noise to the observed next-state **delta**. Sampling with different noise seeds yields multiple plausible futures.

In [ ]:
# Only stochastic/outcome coordinates are learned. Bid, budget, remaining fraction, weekday and quality are deterministic/derived.
CAMPAIGN_DYN_IDX=torch.tensor([1,3,4,5,6],dtype=torch.long); QUERY_DYN_IDX=torch.tensor([0],dtype=torch.long)
x_dynamic_mask=torch.zeros((N,DYN_DIM),dtype=torch.bool); x_dynamic_mask[campaign_gids[:,None],CAMPAIGN_DYN_IDX[None,:]]=True; x_dynamic_mask[query_gids[:,None],QUERY_DYN_IDX[None,:]]=True

def deterministic_feature_transition(x_prev,iv):
    b=x_prev.clone(); c=b[campaign_gids].clone(); budget=torch.expm1(c[:,0]).clamp_min(0); spend=torch.expm1(c[:,1]).clamp_min(0); bid=torch.expm1(c[:,2]).clamp_min(.12)
    bd=torch.tensor(iv.bid_delta,dtype=c.dtype); ud=torch.tensor(iv.budget_delta,dtype=c.dtype)
    nb=torch.maximum(torch.maximum(torch.tensor(50.,dtype=c.dtype),budget*(1+ud)),spend); nbi=torch.clamp(bid*torch.exp(bd),.12,8.)
    c[:,0]=torch.log1p(nb); c[:,2]=torch.log1p(nbi); c[:,7]=(1-spend/nb.clamp_min(1e-6)).clamp(0,1); c[:,8]=torch.remainder(torch.round(c[:,8]*6)+1,7)/6; c[:,9]=x_prev[campaign_gids,9]
    b[campaign_gids]=c; q=b[query_gids].clone(); q[:,8]=torch.remainder(torch.round(q[:,8]*6)+1,7)/6; b[query_gids]=q
    return b

def fit_delta_stats(rows,mask):
    vals={j:[] for j in range(DYN_DIM)}
    for r in rows:
        resid=r["y"]-deterministic_feature_transition(r["x"],r["iv"])
        for j in range(DYN_DIM):
            mj=mask[:,j]
            if mj.any(): vals[j].append(resid[mj,j])
    mean=torch.zeros(DYN_DIM); std=torch.ones(DYN_DIM)
    for j,arr in vals.items():
        if arr:
            v=torch.cat(arr); mean[j]=v.mean(); std[j]=v.std().clamp_min(1e-4)
    return mean,std
x_delta_mean,x_delta_std=fit_delta_stats(x_train,x_dynamic_mask)

def normalize_delta_tensor(delta,mean=None,std=None,mask=None):
    mean=x_delta_mean if mean is None else mean; std=x_delta_std if std is None else std; mask=x_dynamic_mask if mask is None else mask
    out=torch.zeros_like(delta)
    for j in range(DYN_DIM):
        mj=mask[:,j]
        if mj.any(): out[mj,j]=(delta[mj,j]-mean[j])/std[j]
    return out

def denormalize_delta_tensor(delta_n,mean=None,std=None,mask=None):
    mean=x_delta_mean if mean is None else mean; std=x_delta_std if std is None else std; mask=x_dynamic_mask if mask is None else mask
    out=torch.zeros_like(delta_n)
    for j in range(DYN_DIM):
        mj=mask[:,j]
        if mj.any(): out[mj,j]=delta_n[mj,j]*std[j]+mean[j]
    return out

def project_valid_state(x_prev,residual,iv):
    base=deterministic_feature_transition(x_prev,iv); x=base.clone(); x[x_dynamic_mask]=base[x_dynamic_mask]+residual[x_dynamic_mask]
    c=x[campaign_gids].clone(); prev=x_prev[campaign_gids]
    c[:,1]=torch.maximum(c[:,1].clamp_min(0),prev[:,1])  # cumulative spend cannot decrease
    c[:,3]=c[:,3].clamp_min(0); c[:,4:6]=c[:,4:6].clamp(0,1); c[:,6]=c[:,6].clamp(0,50)
    budget=torch.expm1(c[:,0]).clamp_min(1e-6); spend=torch.expm1(c[:,1]).clamp_min(0); c[:,1]=torch.log1p(torch.minimum(spend,budget)); spend=torch.expm1(c[:,1]); c[:,7]=(1-spend/budget).clamp(0,1)
    c[:,2]=base[campaign_gids,2]; c[:,8]=base[campaign_gids,8]; c[:,9]=x_prev[campaign_gids,9]; x[campaign_gids]=c
    q=x[query_gids].clone(); q[:,0]=q[:,0].clamp_min(0); q[:,8]=base[query_gids,8]; x[query_gids]=q
    return x

In [ ]:
class XEdgeAwareLayer(nn.Module):
    def __init__(self,d,heads,n_rel,n_types,dropout=.08):
        super().__init__(); assert d%heads==0; self.d=d; self.h=heads; self.dk=d//heads
        self.q=nn.Linear(d,d,bias=False); self.k=nn.Linear(d,d,bias=False); self.v=nn.Linear(d,d,bias=False); self.rel=nn.Embedding(n_rel,heads); self.gate=nn.Embedding(n_rel,heads); self.st=nn.Embedding(n_types,heads); self.dt=nn.Embedding(n_types,heads); self.edge_weight_scale=nn.Parameter(torch.ones(heads)); self.out=nn.Linear(d,d); self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d); self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(dropout),nn.Linear(4*d,d)); self.drop=nn.Dropout(dropout)
    def forward(self,h,ei,et,nt,ew):
        src,dst=ei; Qv=self.q(h).view(-1,self.h,self.dk); Kv=self.k(h).view(-1,self.h,self.dk); Vv=self.v(h).view(-1,self.h,self.dk); score=(Qv[dst]*Kv[src]).sum(-1)/math.sqrt(self.dk)+self.rel(et)+self.st(nt[src])+self.dt(nt[dst]); score+=torch.log(ew.clamp_min(1e-4))[:,None]*self.edge_weight_scale[None,:]; alpha=segment_softmax(score,dst,h.shape[0]); gate=torch.sigmoid(self.gate(et)); msg=(alpha*gate)[...,None]*Vv[src]; agg=torch.zeros((h.shape[0],self.h,self.dk),device=h.device); agg.index_add_(0,dst,msg); z=self.n1(h+self.drop(self.out(agg.flatten(1)))); return self.n2(z+self.drop(self.ff(z))),alpha
class GraphFlowWorldModel(nn.Module):
    def __init__(self,state_dim=DYN_DIM,sem_dim=X_SEM_DIM,d=X_DMODEL,heads=X_HEADS,layers=X_LAYERS):
        super().__init__(); self.state=nn.Linear(state_dim,d); self.sem=nn.Linear(sem_dim,d); self.type_emb=nn.Embedding(len(X_NODE_TYPES),d); self.action=nn.Linear(4,d,bias=False); self.layers=nn.ModuleList([XEdgeAwareLayer(d,heads,X_NUM_RELATIONS,len(X_NODE_TYPES)) for _ in range(layers)]); self.z_in=nn.Linear(state_dim,d); self.t_embed=nn.Sequential(nn.Linear(1,d),nn.SiLU(),nn.Linear(d,d)); self.velocity=nn.Sequential(nn.Linear(2*d,d),nn.SiLU(),nn.Linear(d,d),nn.SiLU(),nn.Linear(d,state_dim)); self.edge_activation=nn.Sequential(nn.Linear(2*d,d),nn.SiLU(),nn.Linear(d,1))
    def encode(self,x,graph):
        n=graph["n_ext"]; xb=torch.zeros((n,x.shape[1]),device=x.device); xb[:N]=x; sem=graph["semantic"].to(x.device); nt=graph["node_type"].to(x.device); act=graph["action"].to(x.device); h=self.state(xb)+self.sem(sem)+self.type_emb(nt)+self.action(act); att=[]; ei=graph["edge_index"].to(x.device); et=graph["edge_type"].to(x.device); ew=graph["edge_weight"].to(x.device)
        for layer in self.layers: h,a=layer(h,ei,et,nt,ew); att.append(a)
        return h,att
    def flow(self,z_tau,tau,context):
        t=self.t_embed(tau.reshape(1,1)).expand(context.shape[0],-1); return self.velocity(torch.cat([self.z_in(z_tau)+t,context],-1))
    def forward(self,x,z_tau,tau,graph):
        ctx,att=self.encode(x,graph); return {"v":self.flow(z_tau,tau,ctx[:N]),"h":ctx,"attn":att}

def _edge_supervision_pairs(r,max_pairs=16):
    pairs=[]; labels=[]; next_elig=r["next_state"].eligibility
    for ci,qj in r["iv"].add_query_edges[:max_pairs]: pairs.append((ci,qj)); labels.append(float(next_elig[ci,qj]>0))
    for ci,qj in r["iv"].drop_query_edges[:max_pairs]: pairs.append((ci,qj)); labels.append(float(next_elig[ci,qj]>0))
    need=max(2,min(6,max_pairs-len(pairs)))
    for _ in range(need):
        ci=int(x_rng.integers(C)); pos=np.where(next_elig[ci]>0)[0]; neg=np.where((targetable_cq[ci]>0)&(next_elig[ci]==0))[0]
        if len(pos): pairs.append((ci,int(x_rng.choice(pos)))); labels.append(1.)
        if len(neg): pairs.append((ci,int(x_rng.choice(neg)))); labels.append(0.)
    return pairs,labels

def flow_loss(model,r,mask_override=None,mean=None,std=None,input_x=None):
    x=(r["x"] if input_x is None else input_x).to(DEVICE); base=deterministic_feature_transition(r["x"],r["iv"]); residual=r["y"]-base; mask=mask_override if mask_override is not None else x_dynamic_mask; target=normalize_delta_tensor(residual,mean,std,mask).to(DEVICE); graph=build_intervention_graph(r["iv"],r["state"]); z0=torch.randn_like(target); tau=torch.rand((),device=DEVICE); zt=(1-tau)*z0+tau*target; target_v=target-z0; out=model(x,zt,tau,graph); m=mask.to(DEVICE); dyn=F.smooth_l1_loss(out["v"][m],target_v[m]); pairs,labels=_edge_supervision_pairs(r); edge_loss=torch.tensor(0.,device=DEVICE)
    if pairs:
        cg=torch.tensor([node_maps["campaign"][campaign_ids[ci]] for ci,qj in pairs],device=DEVICE); qg=torch.tensor([node_maps["query"][str(query_ids[qj])] for ci,qj in pairs],device=DEVICE); h=out["h"]; logits=model.edge_activation(torch.cat([h[cg],h[qg]],-1)).squeeze(-1); edge_loss=F.binary_cross_entropy_with_logits(logits,torch.tensor(labels,dtype=torch.float32,device=DEVICE))
    return dyn+.06*edge_loss

@torch.no_grad()
def sample_flow_delta(model,x,iv,structure=None,steps=X_FLOW_STEPS,seed=None,mean=None,std=None,mask=None):
    mask=x_dynamic_mask if mask is None else mask
    if seed is not None: gen=torch.Generator(device=DEVICE); gen.manual_seed(int(seed)); z=torch.randn((N,DYN_DIM),generator=gen,device=DEVICE)
    else: z=torch.randn((N,DYN_DIM),device=DEVICE)
    z[~mask.to(DEVICE)]=0; graph=build_intervention_graph(iv,structure); dt=1./steps
    for k in range(steps):
        tau=torch.tensor((k+.5)/steps,device=DEVICE); out=model(x.to(DEVICE),z,tau,graph); z=z+dt*out["v"]; z[~mask.to(DEVICE)]=0
    return denormalize_delta_tensor(z.cpu(),mean,std,mask),out

## X6. Deep flow ensemble, epistemic disagreement, and OOD support score

Each ensemble member is trained with a different seed/bootstrap ordering. Planning uncertainty combines:
- future sample dispersion from flow noise;
- ensemble disagreement (epistemic);
- graph-state OOD distance to train embeddings.

In [ ]:
def train_flow_member(seed_offset=0,epochs=X_TRAIN_EPOCHS):
    torch.manual_seed(X_SEED+seed_offset); m=GraphFlowWorldModel().to(DEVICE); optx=torch.optim.AdamW(m.parameters(),lr=1.5e-3,weight_decay=2e-4); best=float("inf"); best_sd=None; patience=0
    for ep in range(epochs):
        m.train(); vals=[]
        for idx in x_rng.permutation(len(x_train)):
            loss=flow_loss(m,x_train[int(idx)]); optx.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1.); optx.step(); vals.append(float(loss.detach()))
        m.eval(); vv=[]
        with torch.no_grad():
            for r in x_val: vv.append(float(flow_loss(m,r)))
        va=float(np.mean(vv)) if vv else float(np.mean(vals)); print(f"member={seed_offset} epoch={ep:02d} train={np.mean(vals):.4f} val={va:.4f}")
        if va<best-1e-4: best=va; patience=0; best_sd={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
        else:
            patience+=1
            if patience>=3: break
    if best_sd is None: raise RuntimeError("Flow model produced no checkpoint")
    m.load_state_dict(best_sd); return m
flow_ensemble=[train_flow_member(k) for k in range(X_ENSEMBLE_SIZE)]
@torch.no_grad()
def context_embedding(model,x,iv,structure=None):
    h,_=model.encode(x.to(DEVICE),build_intervention_graph(iv,structure)); action_nodes=h[N:]; pool=torch.cat([h[campaign_gids.to(DEVICE)],action_nodes],0) if len(action_nodes) else h[campaign_gids.to(DEVICE)]; return pool.mean(0).cpu()
_zero_iv=Intervention(np.zeros(C,np.float32),np.zeros(C,np.float32))
_train_ctx=torch.stack([context_embedding(flow_ensemble[0],r["x"],r["iv"],r["state"]) for r in x_train]); ctx_mean=_train_ctx.mean(0); ctx_std=_train_ctx.std(0).clamp_min(1e-4)
def graph_ood_score(x,iv,structure=None):
    z=context_embedding(flow_ensemble[0],x,iv,structure); return float(torch.sqrt((((z-ctx_mean)/ctx_std)**2).mean()))
@torch.no_grad()
def ensemble_futures(x,iv,structure=None,k_per_model=3):
    futures=[]
    for mi,m in enumerate(flow_ensemble):
        for k in range(k_per_model):
            d,_=sample_flow_delta(m,x,iv,structure,seed=10000+mi*100+k); futures.append(project_valid_state(x,d,iv))
    S=torch.stack(futures); return S,S.std(0),graph_ood_score(x,iv,structure)

## X7. Independent advertiser opponent policies

Non-focal advertisers are no longer controlled by the focal planner. A latent advertiser policy predicts each campaign's bid/budget response from its state plus advertiser identity and market competition context.

In [ ]:
class OpponentPolicy(nn.Module):
    def __init__(self,n_adv=A,d=64):
        super().__init__(); self.adv=nn.Embedding(max(1,n_adv),16); self.net=nn.Sequential(nn.Linear(DYN_DIM+16+4+2,d),nn.SiLU(),nn.Linear(d,d),nn.SiLU(),nn.Linear(d,4))
    def forward(self,campaign_x,adv_idx,context,focal_signal):
        z=torch.cat([campaign_x,self.adv(adv_idx),context,focal_signal],-1); out=self.net(z); return out[:,:2],out[:,2:].clamp(-5,1)
def opponent_batch(r,focal_signal=None):
    cx=r["x"][campaign_gids].float(); adv=torch.tensor(campaign_adv_idx,dtype=torch.long); roas=cx[:,6]; spend=torch.expm1(cx[:,1]).clamp_min(0); rem=cx[:,7]; ctx=torch.stack([roas.mean().expand_as(roas),roas.std().expand_as(roas),spend.mean().expand_as(roas),rem.mean().expand_as(roas)],1); y=torch.tensor(np.c_[r["iv"].bid_delta,r["iv"].budget_delta],dtype=torch.float32)
    if focal_signal is None:
        fb=float(r["iv"].bid_delta[focal_campaigns].mean()) if len(focal_campaigns) else 0.; fu=float(r["iv"].budget_delta[focal_campaigns].mean()) if len(focal_campaigns) else 0.; focal_signal=torch.tensor([fb,fu]).repeat(C,1)
    return cx,adv,ctx,focal_signal.float(),y
opponent=OpponentPolicy().to(DEVICE); oop=torch.optim.AdamW(opponent.parameters(),lr=2e-3,weight_decay=1e-4)
for ep in range(12):
    opponent.train(); ls=[]
    for r in x_train:
        cx,ai,ctx,fs,y=opponent_batch(r); mu,lv=opponent(cx.to(DEVICE),ai.to(DEVICE),ctx.to(DEVICE),fs.to(DEVICE)); y=y.to(DEVICE); nonf=torch.ones(C,dtype=torch.bool,device=DEVICE); nonf[torch.tensor(focal_campaigns,dtype=torch.long,device=DEVICE)]=False; loss=(.5*(lv+(y-mu).pow(2)*torch.exp(-lv)))[nonf].mean() if nonf.any() else (.5*(lv+(y-mu).pow(2)*torch.exp(-lv))).mean(); oop.zero_grad(set_to_none=True); loss.backward(); oop.step(); ls.append(float(loss.detach()))
    print("opponent",ep,np.mean(ls))
@torch.no_grad()
def sample_opponent_action(x,base_iv=None,temperature=.7):
    fb=float(base_iv.bid_delta[focal_campaigns].mean()) if base_iv is not None and len(focal_campaigns) else 0.; fu=float(base_iv.budget_delta[focal_campaigns].mean()) if base_iv is not None and len(focal_campaigns) else 0.; fs=torch.tensor([fb,fu]).repeat(C,1); dummy={"x":x,"iv":base_iv or _zero_iv}; cx,ai,ctx,_,_=opponent_batch(dummy,fs); mu,lv=opponent(cx.to(DEVICE),ai.to(DEVICE),ctx.to(DEVICE),fs.to(DEVICE)); sd=torch.exp(.5*lv)*temperature; act=(mu+sd*torch.randn_like(mu)).cpu().numpy(); bid=np.clip(act[:,0],-.22,.22).astype(np.float32); bud=np.clip(act[:,1],-.15,.15).astype(np.float32)
    if base_iv is not None: bid[focal_campaigns]=base_iv.bid_delta[focal_campaigns]; bud[focal_campaigns]=base_iv.budget_delta[focal_campaigns]
    return Intervention(bid,bud,list(base_iv.add_query_edges) if base_iv else [],list(base_iv.drop_query_edges) if base_iv else [],list(base_iv.expand_market) if base_iv else [])

## X8. Meaningful cross-market/channel optimal transport

Transport sources are `(advertiser, channel, market)` budget pools; destinations are `(campaign, query-cluster, market)` opportunity units. Costs differ by source/destination because they include market mismatch, channel friction, saturation, uncertainty, and predicted marginal return.

In [ ]:
CHANNELS=["sponsored_products","display","video"]; channel_friction={"sponsored_products":0.,"display":.10,"video":.18}
def structured_budget_transport(xs,uncertainty=None,total_fraction=.04,eps=.18,max_dest_per_adv=96):
    """Advertiser-isolated OT from current remaining-budget pools to campaign/query-cluster/market opportunities."""
    s=xs.base; roas=safe_ratio(s.cum_revenue,np.maximum(s.spend,1e-4)); sat=np.clip(safe_ratio(s.spend,s.budget),0,1)
    if uncertainty is None: uncertainty=np.full(C,.15,np.float32)
    camp_alloc=np.zeros(C,np.float32); blocks=[]; total_money=0.
    for a in range(A):
        camps=np.where(campaign_adv_idx==a)[0]
        if not len(camps): continue
        sources=[]; supply=[]
        for mk,m in enumerate(MARKETS):
            local=camps[campaign_locale[camps]==mk]
            if not len(local): continue
            rem=float(np.maximum(s.budget[local]-s.spend[local],0).sum())
            if rem<=1e-6: continue
            for ch in CHANNELS: sources.append((a,ch,mk)); supply.append(max(1e-6,rem*total_fraction/len(CHANNELS)))
        if not sources: continue
        dest=[]; desir=[]
        for ci in camps:
            for mk in range(len(MARKETS)):
                qs=np.where((query_locale==mk)&targetable_cq[ci])[0]
                if not len(qs): continue
                for qc in sorted({int(query_cluster[q]) for q in qs})[:4]:
                    members=np.where((query_cluster==qc)&(query_locale==mk))[0]; rel=float(effective_rel_cq[ci,members].mean()) if len(members) else 0.; access_bonus=.15 if xs.market_access[ci,mk]>0 else 0.; dest.append((ci,qc,mk)); desir.append(max(.01,rel*(max(float(roas[ci]),.1)+access_bonus)))
        if not dest: continue
        if len(dest)>max_dest_per_adv:
            keep=np.argsort(desir)[-max_dest_per_adv:]; dest=[dest[i] for i in keep]; desir=[desir[i] for i in keep]
        supply=np.asarray(supply,np.float32); block_money=float(supply.sum()); total_money+=block_money; supply/=supply.sum(); demand=np.asarray(desir,np.float32); demand/=demand.sum(); cost=np.zeros((len(sources),len(dest)),np.float32)
        for i,(aa,ch,sm) in enumerate(sources):
            for j,(ci,qc,dm) in enumerate(dest):
                market_mismatch=0. if sm==dm else .35; expansion=.18 if xs.market_access[ci,dm]==0 else 0.; cost[i,j]=-float(roas[ci])+.7*float(uncertainty[ci])+.35*float(sat[ci])+market_mismatch+expansion+channel_friction[ch]-.25*float(desir[j])
        T=log_sinkhorn(torch.tensor(-cost/eps),torch.log(torch.tensor(supply).clamp_min(1e-12)),torch.log(torch.tensor(demand).clamp_min(1e-12)),iters=100).numpy(); money=T*block_money
        for j,(ci,qc,dm) in enumerate(dest): camp_alloc[ci]+=money[:,j].sum()
        blocks.append({"advertiser":a,"sources":sources,"destinations":dest,"cost":cost,"transport":money})
    return camp_alloc,blocks,{"total_transport_money":total_money,"block_count":len(blocks)}

## X9. Risk-sensitive hierarchical Graph-MPPI

Stage 1 selects high-value focal campaigns. Stage 2 optimizes only those campaign controls. Non-focal advertisers are sampled from their learned opponent policies. Candidate trajectories are evaluated over multiple flow futures.

Objective combines expected value, lower-tail CVaR, ensemble/trajectory dispersion, OOD support, and action magnitude.

In [ ]:
def x_reward(x):
    c=x[campaign_gids]; fc=torch.tensor(focal_campaigns if len(focal_campaigns) else [0],dtype=torch.long); z=c[fc]; spend=torch.expm1(z[:,1]).clamp_min(0); roas=z[:,6].clamp(0,50); cvr=z[:,5].clamp(0,1); remain=z[:,7].clamp(0,1); return float((.58*roas+.20*cvr+.12*remain-.000015*spend).mean())
def intervention_value_scores(x):
    c=x[campaign_gids]; return (.55*c[:,6]+.20*c[:,5]+.15*c[:,7]).numpy()
def sample_structural_moves(active_ci,structure,prob=.28):
    elig,access=_structure_arrays(structure); out={"add_query_edges":[],"drop_query_edges":[],"expand_market":[]}
    for ci in active_ci:
        if x_rng.random()>prob: continue
        addcand=np.where(targetable_cq[ci]&(elig[ci]==0))[0]; live=np.where(elig[ci]>0)[0]
        if len(addcand) and x_rng.random()<.55:
            q=int(addcand[np.argmax(effective_rel_cq[ci,addcand])]); out["add_query_edges"].append((int(ci),q)); mk=query_locale[q]
            if access[ci,mk]==0: out["expand_market"].append((int(ci),MARKETS[int(mk)]))
        if len(live) and x_rng.random()<.20: out["drop_query_edges"].append((int(ci),int(x_rng.choice(live))))
    return out
def _iv_from_control(control,active_ci,structural=None):
    iv=Intervention(np.zeros(C,np.float32),np.zeros(C,np.float32));
    for j,ci in enumerate(active_ci): iv.bid_delta[ci]=float(np.clip(control[j,0],-.22,.22)); iv.budget_delta[ci]=float(np.clip(control[j,1],-.15,.15))
    if structural: iv.add_query_edges=list(structural.get("add_query_edges",[])); iv.drop_query_edges=list(structural.get("drop_query_edges",[])); iv.expand_market=list(structural.get("expand_market",[]))
    return iv
def merge_focal_control(x,control,active_ci,structural=None):
    focal=_iv_from_control(control,active_ci,structural); return sample_opponent_action(x,focal)
def _apply_structure_arrays(elig,access,iv):
    ne=elig.copy(); na=access.copy()
    for ci,qj in iv.add_query_edges:
        if 0<=ci<C and 0<=qj<Q and targetable_cq[ci,qj]: ne[ci,qj]=1.
    for ci,qj in iv.drop_query_edges:
        if 0<=ci<C and 0<=qj<Q: ne[ci,qj]=0.
    for ci,loc in iv.expand_market:
        if 0<=ci<C and loc in market_to_k: na[ci,market_to_k[loc]]=1.
    return ne,na
@torch.no_grad()
def generative_rollout(x0,controls,active_ci,structure,structural_seq=None,futures_per_step=2):
    e0,a0=_structure_arrays(structure); particles=[(x0.clone(),e0.copy(),a0.copy()) for _ in range(max(2,futures_per_step*len(flow_ensemble)))]; cumulative=np.zeros(len(particles),np.float32); oods=np.zeros(len(particles),np.float32); structural_seq=structural_seq or [None]*len(controls)
    for h,ctrl in enumerate(controls):
        nxt=[]
        for p,(x,elig,access) in enumerate(particles):
            st={"eligibility":elig,"market_access":access}; iv=merge_focal_control(x,ctrl,active_ci,structural_seq[h]); m=flow_ensemble[p%len(flow_ensemble)]; d,_=sample_flow_delta(m,x,iv,st,seed=400000+h*1000+p); nx=project_valid_state(x,d,iv); ne,na=_apply_structure_arrays(elig,access,iv); nxt.append((nx,ne,na)); cumulative[p]+=x_reward(nx); oods[p]+=graph_ood_score(x,iv,st)
        particles=nxt
    stack=torch.stack([p[0] for p in particles]); disagreement=float(stack[:,campaign_gids].std(0).mean()); return cumulative,stack,disagreement,float(oods.mean()/max(1,len(controls)))
def cvar_lower(values,alpha=.25):
    v=np.sort(np.asarray(values)); k=max(1,int(np.ceil(alpha*len(v)))); return float(v[:k].mean())
def graph_mppi(x_now,structure=None,horizon=3,topk=6,samples=32,iters=3,temp=.7,noise=.07,cvar_alpha=.25,risk_lambda=.20):
    structure=structure or {"eligibility":base_eligibility,"market_access":initial_market_access}; scores0=intervention_value_scores(x_now); fc=focal_campaigns if len(focal_campaigns) else np.array([0]); order=fc[np.argsort(scores0[fc])]; active_ci=order[-min(topk,len(order)):]; mean=np.zeros((horizon,len(active_ci),2),np.float32); best=None
    for it in range(iters):
        ctrls=[]; scores=[]
        for n in range(samples):
            u=mean+x_rng.normal(0,noise,size=mean.shape).astype(np.float32); u[:,:,0]=np.clip(u[:,:,0],-.22,.22); u[:,:,1]=np.clip(u[:,:,1],-.15,.15); _e,_a=_structure_arrays(structure); _e=_e.copy(); _a=_a.copy(); structural=[]
            for h in range(horizon):
                mv=sample_structural_moves(active_ci,{"eligibility":_e,"market_access":_a},prob=.30 if h==0 else .08); structural.append(mv); tmp=_iv_from_control(np.zeros((len(active_ci),2),np.float32),active_ci,mv); _e,_a=_apply_structure_arrays(_e,_a,tmp)
            returns,parts,dis,ood=generative_rollout(x_now,u,active_ci,structure,structural,2); cv=cvar_lower(returns,cvar_alpha); avg=float(np.mean(returns)); score=avg+.45*cv-risk_lambda*dis-.08*ood-.10*float(np.square(u).mean()); ctrls.append(u); scores.append(score)
            if best is None or score>best["score"]: best={"score":score,"control":u.copy(),"active_ci":active_ci.copy(),"structural":structural,"diagnostics":(avg,cv,dis,ood)}
        ss=np.asarray(scores); w=np.exp((ss-ss.max())/max(temp,1e-4)); w/=w.sum()+1e-9; mean=np.tensordot(w,np.stack(ctrls),axes=(0,0)); print(f"MPPI iter={it} best={ss.max():.4f} mean={ss.mean():.4f}")
    first=merge_focal_control(x_now,best["control"][0],best["active_ci"],best["structural"][0]); return {**best,"first_intervention":first}

## X10. CVaR + OOD + support-aware verifier

The verifier uses generated futures, not a single scalar sigma. It can `EXECUTE`, `REVISE`, or `ABSTAIN` based on lower-tail return, graph support, trajectory disagreement, finite-state invariants, and action limits.

In [ ]:
def verify_intervention(x,iv,structure=None,cvar_floor=None,ood_limit=3.5,disagreement_limit=1.5,ensemble=None):
    if np.max(np.abs(iv.bid_delta))>.22: return "REVISE",{"reason":"bid step"}
    if np.max(np.abs(iv.budget_delta))>.15: return "REVISE",{"reason":"budget step"}
    futures,disp,ood=ensemble_futures(x,iv,structure,k_per_model=3); vals=np.array([x_reward(f) for f in futures],np.float32); cv=cvar_lower(vals,.25); dis=float(disp[campaign_gids].mean())
    if not torch.isfinite(futures).all(): return "ABSTAIN",{"reason":"non-finite future"}
    if ood>ood_limit: return "ABSTAIN",{"reason":"OOD","ood":ood,"cvar":cv,"disagreement":dis}
    if dis>disagreement_limit: return "ABSTAIN",{"reason":"trajectory disagreement","ood":ood,"cvar":cv,"disagreement":dis}
    if cvar_floor is not None and cv<cvar_floor: return "REVISE",{"reason":"CVaR floor","ood":ood,"cvar":cv,"disagreement":dis}
    return "EXECUTE",{"ood":ood,"cvar":cv,"mean":float(vals.mean()),"disagreement":dis}

## X11. Offline policy evaluation: SNIPS + doubly robust

OPE is performed on held-out logged interventions. To avoid unstable densities in the full campaign action space, evaluation uses the **focal advertiser's 2-D aggregate action** (mean bid change, mean budget change), with clipped importance ratios.

In [ ]:
def focal_action_vector(iv):
    fc=focal_campaigns if len(focal_campaigns) else np.arange(min(1,C)); return np.concatenate([iv.bid_delta[fc],iv.budget_delta[fc]]).astype(np.float32)
def state_summary(x):
    c=x[campaign_gids]; fc=torch.tensor(focal_campaigns if len(focal_campaigns) else [0],dtype=torch.long); z=c[fc]; return torch.tensor([float(z[:,6].mean()),float(z[:,5].mean()),float(z[:,7].mean()),float(torch.expm1(z[:,1]).mean())],dtype=torch.float32)
def _atanh_clip(y): return np.arctanh(np.clip(y,-.999999,.999999))
def bounded_logpdf(action,mean,sd,limit):
    action=np.asarray(action); mean=np.asarray(mean); y=action/limit; z=limit*_atanh_clip(y); logn=-.5*((z-mean)/sd)**2-np.log(sd)-.5*np.log(2*np.pi); jac=np.log(limit)+np.log(np.maximum(1-y*y,1e-8)); return float((logn-jac).sum())
def behavior_logprob_focal(r):
    fc=focal_campaigns if len(focal_campaigns) else np.array([0]); bm,um=_behavior_means(r["state"]); return bounded_logpdf(r["iv"].bid_delta[fc],bm[fc],.07,.22)+bounded_logpdf(r["iv"].budget_delta[fc],um[fc],.045,.15)
_action_dim=2*max(1,len(focal_campaigns))
class OutcomeModel(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Linear(4+_action_dim,64),nn.SiLU(),nn.Linear(64,48),nn.SiLU(),nn.Linear(48,1))
    def forward(self,s,a): return self.net(torch.cat([s,a],-1)).squeeze(-1)
outcome_model=OutcomeModel().to(DEVICE); oo=torch.optim.AdamW(outcome_model.parameters(),lr=2e-3)
_train_ope=[r for r in x_train if not (r["iv"].add_query_edges or r["iv"].drop_query_edges or r["iv"].expand_market)]
for ep in range(30):
    ls=[]
    for r in _train_ope:
        s=state_summary(r["x"]).to(DEVICE); a=torch.tensor(focal_action_vector(r["iv"]),device=DEVICE); y=torch.tensor(r["reward"],dtype=torch.float32,device=DEVICE); pred=outcome_model(s,a); loss=F.smooth_l1_loss(pred,y); oo.zero_grad(set_to_none=True); loss.backward(); oo.step(); ls.append(float(loss.detach()))
    if ep%10==0 and ls: print("outcome",ep,np.mean(ls))
@torch.no_grad()
def target_action_vector(r):
    plan=graph_mppi(r["x"],r["state"],horizon=2,topk=min(4,max(1,len(focal_campaigns))),samples=12,iters=2); return focal_action_vector(plan["first_intervention"])
def target_logprob(observed,target,sd_bid=.04,sd_budget=.03):
    fc=max(1,len(focal_campaigns)); return bounded_logpdf(observed[:fc],target[:fc],sd_bid,.22)+bounded_logpdf(observed[fc:],target[fc:],sd_budget,.15)
def ope_snips_dr(rows,clip=20.):
    rows=[r for r in rows if not (r["iv"].add_query_edges or r["iv"].drop_query_edges or r["iv"].expand_market)]; weights=[]; rewards=[]; dr=[]
    for r in rows:
        s=state_summary(r["x"]); a=focal_action_vector(r["iv"]); ta=target_action_vector(r); logw=target_logprob(a,ta)-behavior_logprob_focal(r); w=float(np.clip(np.exp(np.clip(logw,-12,12)),0,clip))
        with torch.no_grad(): ql=float(outcome_model(s.to(DEVICE),torch.tensor(a,device=DEVICE))); qt=float(outcome_model(s.to(DEVICE),torch.tensor(ta,device=DEVICE)))
        weights.append(w); rewards.append(float(r["reward"])); dr.append(qt+w*(float(r["reward"])-ql))
    if not weights: return {"note":"No non-structural rows available for continuous-action OPE"}
    w=np.asarray(weights); y=np.asarray(rewards); ess=float(w.sum()**2/max((w*w).sum(),1e-30)); overlap_ok=bool(w.sum()>1e-6 and ess>=max(2.,.10*len(w)))
    if not overlap_ok: return {"SNIPS":None,"DR":None,"ESS":ess,"n":len(w),"scope":"continuous focal actions only","status":"INSUFFICIENT_POLICY_OVERLAP"}
    return {"SNIPS":float((w*y).sum()/w.sum()),"DR":float(np.mean(dr)),"ESS":ess,"n":len(w),"scope":"continuous focal actions only","status":"OK"}
ope_result=ope_snips_dr(x_test[:min(12,len(x_test))]) if x_test else {}; print(ope_result)

## X12. US+ES → JP cold-start transfer

Japan remains in the semantic/structural graph, but its dynamic nodes are excluded from world-model transition supervision. Evaluation then measures zero-shot next-state prediction and support-aware abstention on JP campaigns/queries.

In [ ]:
jp_campaign_idx=np.where(campaign_seed.locale.astype(str).to_numpy()=="jp")[0]; jp_query_idx=np.where(query_tbl.locale.astype(str).to_numpy()=="jp")[0]
jp_input_mask=torch.zeros((N,DYN_DIM),dtype=torch.bool)
if len(jp_campaign_idx): jp_input_mask[campaign_gids[jp_campaign_idx[:,None]],torch.arange(9)[None,:]]=True
if len(jp_query_idx): jp_input_mask[query_gids[jp_query_idx[:,None]],torch.tensor([0,8])[None,:]]=True
jp_target_mask=x_dynamic_mask & jp_input_mask; train_us_es_mask=x_dynamic_mask & (~jp_input_mask); cold_mean,cold_std=fit_delta_stats(x_train,train_us_es_mask)
# Type-matched US/ES fill values: JP campaign history is replaced by non-JP campaign statistics, JP queries by non-JP query statistics.
nonjp_camp=np.setdiff1d(np.arange(C),jp_campaign_idx); nonjp_query=np.setdiff1d(np.arange(Q),jp_query_idx)
camp_fill=torch.zeros(DYN_DIM); query_fill=torch.zeros(DYN_DIM)
for j in range(DYN_DIM):
    cv=[]; qv=[]
    for r in x_train:
        if len(nonjp_camp): cv.append(r["x"][campaign_gids[nonjp_camp],j])
        if len(nonjp_query): qv.append(r["x"][query_gids[nonjp_query],j])
    if cv: camp_fill[j]=torch.cat(cv).mean()
    if qv: query_fill[j]=torch.cat(qv).mean()
def mask_jp_input(x):
    z=x.clone()
    if len(jp_campaign_idx): z[campaign_gids[jp_campaign_idx]]=camp_fill
    if len(jp_query_idx): z[query_gids[jp_query_idx]]=query_fill
    # preserve static campaign quality and deterministic market semantics outside dynamic history
    if len(jp_campaign_idx): z[campaign_gids[jp_campaign_idx],9]=x[campaign_gids[jp_campaign_idx],9]
    return z
class ColdStartMLP(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Linear(DYN_DIM+4,96),nn.SiLU(),nn.Linear(96,DYN_DIM))
    def forward(self,x,a): return self.net(torch.cat([x,a],-1))
cold_mlp=ColdStartMLP().to(DEVICE); cmopt=torch.optim.AdamW(cold_mlp.parameters(),lr=2e-3)
for ep in range(12):
    ls=[]
    for r in x_train:
        x=mask_jp_input(r["x"]).to(DEVICE); residual=r["y"]-deterministic_feature_transition(r["x"],r["iv"]); target=normalize_delta_tensor(residual,cold_mean,cold_std,train_us_es_mask).to(DEVICE); a=torch.zeros((N,4),device=DEVICE); a[campaign_gids,0]=torch.tensor(r["iv"].bid_delta,device=DEVICE); a[campaign_gids,1]=torch.tensor(r["iv"].budget_delta,device=DEVICE); pred=cold_mlp(x,a); m=train_us_es_mask.to(DEVICE); loss=F.smooth_l1_loss(pred[m],target[m]); cmopt.zero_grad(set_to_none=True); loss.backward(); cmopt.step(); ls.append(float(loss.detach()))
    if ep%4==0: print("cold MLP",ep,np.mean(ls))
def train_cold_flow(epochs=max(6,X_TRAIN_EPOCHS//2)):
    m=GraphFlowWorldModel().to(DEVICE); op=torch.optim.AdamW(m.parameters(),lr=1.5e-3,weight_decay=2e-4); best=float("inf"); best_sd=None
    for ep in range(epochs):
        m.train(); tr=[]
        for r in x_train:
            loss=flow_loss(m,r,mask_override=train_us_es_mask,mean=cold_mean,std=cold_std,input_x=mask_jp_input(r["x"])); op.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1.); op.step(); tr.append(float(loss.detach()))
        m.eval(); vv=[]
        with torch.no_grad():
            for r in x_val: vv.append(float(flow_loss(m,r,mask_override=train_us_es_mask,mean=cold_mean,std=cold_std,input_x=mask_jp_input(r["x"]))))
        va=float(np.mean(vv)) if vv else float(np.mean(tr)); print("cold flow",ep,np.mean(tr),va)
        if va<best: best=va; best_sd={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
    m.load_state_dict(best_sd); return m
cold_flow=train_cold_flow()
@torch.no_grad()
def jp_zero_shot_metrics(rows):
    if not jp_target_mask.any(): return {"note":"No JP dynamic nodes in sampled ESCI graph"}
    if not rows: return {"note":"No held-out episodes available"}
    mlp_err=[]; flow_err=[]; uncertainties=[]
    for ii,r in enumerate(rows):
        xin=mask_jp_input(r["x"]); target=(r["y"]-deterministic_feature_transition(r["x"],r["iv"]))[jp_target_mask]; a=torch.zeros((N,4),device=DEVICE); a[campaign_gids,0]=torch.tensor(r["iv"].bid_delta,device=DEVICE); a[campaign_gids,1]=torch.tensor(r["iv"].budget_delta,device=DEVICE); predn=cold_mlp(xin.to(DEVICE),a).cpu(); pred=denormalize_delta_tensor(predn,cold_mean,cold_std,x_dynamic_mask)[jp_target_mask]; mlp_err.append(float(torch.sqrt(((pred-target)**2).mean())))
        draws=[]
        for k in range(4): draws.append(sample_flow_delta(cold_flow,xin,r["iv"],r["state"],seed=7000+ii*10+k,mean=cold_mean,std=cold_std,mask=x_dynamic_mask)[0])
        D=torch.stack(draws); flow_err.append(float(torch.sqrt(((D.mean(0)[jp_target_mask]-target)**2).mean()))); uncertainties.append(float(D[:,jp_target_mask].std(0).mean()))
    return {"MLP_JP_residual_RMSE":float(np.mean(mlp_err)),"GraphFlow_JP_residual_RMSE":float(np.mean(flow_err)),"JP_trajectory_disagreement":float(np.mean(uncertainties)),"protocol":"JP dynamic history/targets/statistics withheld; JP static semantics/catalog graph retained"}
jp_results=jp_zero_shot_metrics(x_test); print(jp_results)

## X13. Controlled ablations: MLP → GraphSAGE → relation GTN → generative GraphWorld-X

All baselines below train on the same GSP logged transitions and typed dynamic-delta target. This avoids comparing models trained on different simulators.

In [ ]:
class DeltaMLP(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Linear(DYN_DIM+2,96),nn.SiLU(),nn.Linear(96,96),nn.SiLU(),nn.Linear(96,DYN_DIM))
    def forward(self,x,iv):
        a=torch.zeros((N,2),device=x.device); a[campaign_gids,0]=torch.tensor(iv.bid_delta,device=x.device); a[campaign_gids,1]=torch.tensor(iv.budget_delta,device=x.device); return self.net(torch.cat([x,a],-1))
class DeltaGraphSAGE(nn.Module):
    def __init__(self,d=96): super().__init__(); self.inp=nn.Linear(DYN_DIM,d); self.self1=nn.Linear(d,d); self.neigh1=nn.Linear(d,d); self.self2=nn.Linear(d,d); self.neigh2=nn.Linear(d,d); self.out=nn.Linear(d,DYN_DIM)
    def aggregate(self,h):
        src,dst=edge_index_d; agg=torch.zeros_like(h); deg=torch.zeros((N,1),device=h.device); agg.index_add_(0,dst,h[src]); deg.index_add_(0,dst,torch.ones((len(dst),1),device=h.device)); return agg/deg.clamp_min(1)
    def forward(self,x,iv): h=F.silu(self.inp(x)); h=F.silu(self.self1(h)+self.neigh1(self.aggregate(h))); h=F.silu(self.self2(h)+self.neigh2(self.aggregate(h))); return self.out(h)
class DeterministicRelationGTN(nn.Module):
    def __init__(self,d=96):
        super().__init__(); self.state=nn.Linear(DYN_DIM,d); self.sem=nn.Linear(X_SEM_DIM,d); self.typ=nn.Embedding(len(X_NODE_TYPES),d); self.layers=nn.ModuleList([XEdgeAwareLayer(d,4,X_NUM_RELATIONS,len(X_NODE_TYPES)) for _ in range(2)]); self.out=nn.Linear(d,DYN_DIM)
    def forward(self,x,iv,structure=None):
        graph=build_intervention_graph(iv,structure); n=graph["n_ext"]; xb=torch.zeros((n,DYN_DIM),device=x.device); xb[:N]=x; h=self.state(xb)+self.sem(graph["semantic"].to(x.device))+self.typ(graph["node_type"].to(x.device));
        for layer in self.layers: h,_=layer(h,graph["edge_index"].to(x.device),graph["edge_type"].to(x.device),graph["node_type"].to(x.device),graph["edge_weight"].to(x.device))
        return self.out(h[:N])
def train_delta_baseline(m,epochs=6):
    m=m.to(DEVICE); op=torch.optim.AdamW(m.parameters(),lr=2e-3,weight_decay=1e-4); mask=x_dynamic_mask.to(DEVICE)
    for ep in range(epochs):
        for r in x_train:
            target=normalize_delta_tensor(r["y"]-deterministic_feature_transition(r["x"],r["iv"])).to(DEVICE); pred=m(r["x"].to(DEVICE),r["iv"],r["state"]) if isinstance(m,DeterministicRelationGTN) else m(r["x"].to(DEVICE),r["iv"]); loss=F.smooth_l1_loss(pred[mask],target[mask]); op.zero_grad(set_to_none=True); loss.backward(); op.step()
    return m
@torch.no_grad()
def delta_rmse(m,rows):
    vals=[]
    for r in rows:
        pn=(m(r["x"].to(DEVICE),r["iv"],r["state"]) if isinstance(m,DeterministicRelationGTN) else m(r["x"].to(DEVICE),r["iv"])).cpu(); p=denormalize_delta_tensor(pn); y=r["y"]-deterministic_feature_transition(r["x"],r["iv"]); vals.append(float(torch.sqrt(((p[x_dynamic_mask]-y[x_dynamic_mask])**2).mean())))
    return float(np.mean(vals)) if vals else float("nan")
delta_mlp=train_delta_baseline(DeltaMLP(),4 if X_FAST_DEMO else 10); delta_sage=train_delta_baseline(DeltaGraphSAGE(),4 if X_FAST_DEMO else 10); delta_gtn=train_delta_baseline(DeterministicRelationGTN(),4 if X_FAST_DEMO else 10)
@torch.no_grad()
def flow_ensemble_rmse(rows):
    vals=[]
    for ii,r in enumerate(rows):
        ds=[sample_flow_delta(m,r["x"],r["iv"],r["state"],seed=90000+ii*10+k)[0] for k,m in enumerate(flow_ensemble)]; p=torch.stack(ds).mean(0); y=r["y"]-deterministic_feature_transition(r["x"],r["iv"]); vals.append(float(torch.sqrt(((p[x_dynamic_mask]-y[x_dynamic_mask])**2).mean())))
    return float(np.mean(vals)) if vals else float("nan")
ablation_results=pd.DataFrame([{"model":"MLP","test_dynamic_RMSE":delta_rmse(delta_mlp,x_test)},{"model":"GraphSAGE","test_dynamic_RMSE":delta_rmse(delta_sage,x_test)},{"model":"Relation-GTN","test_dynamic_RMSE":delta_rmse(delta_gtn,x_test)},{"model":"GraphFlow-X","test_dynamic_RMSE":flow_ensemble_rmse(x_test)}]); ablation_results

## X13. Planner/verifier tool hierarchy and MCP-ready schemas

The tools intentionally separate planning from execution. A language-model agent can ask for counterfactual analysis, but only the verifier can authorize execution.

In [ ]:
class GraphWorldAdsXTools:
    def __init__(self,xstate=None): self.state=xstate or initial_x_state()
    def inspect(self): return {"base_nodes":N,"base_edges":int(edge_index.shape[1]),"campaigns":C,"advertisers":A,"queries":Q,"markets":MARKETS,"semantic_backend":semantic_meta["backend"]}
    def optimize_transport(self):
        alloc,blocks,meta=structured_budget_transport(self.state); return {"campaign_allocation":alloc,"transport_blocks":blocks,"meta":meta}
    def propose(self,horizon=3):
        x=x_state_to_features(self.state); return graph_mppi(x,self.state,horizon=horizon,topk=min(6,max(1,len(focal_campaigns))),samples=24 if X_FAST_DEMO else 64,iters=3)
    def simulate(self,iv,k=6):
        x=x_state_to_features(self.state); futures,disp,ood=ensemble_futures(x,iv,self.state,k_per_model=max(1,k//len(flow_ensemble))); return {"mean_reward":float(np.mean([x_reward(f) for f in futures])),"cvar":cvar_lower([x_reward(f) for f in futures]),"ood":ood,"disagreement":float(disp[campaign_gids].mean())}
    def verify(self,iv): return verify_intervention(x_state_to_features(self.state),iv,self.state)
    def execute_in_simulator(self,iv):
        verdict,diag=self.verify(iv)
        if verdict!="EXECUTE": return {"status":verdict,"diagnostics":diag}
        self.state=market_step_gsp(self.state,iv,True); return {"status":"EXECUTED_IN_CONTROLLED_SIMULATOR","diagnostics":diag}
MCP_TOOL_SCHEMAS={"inspect_market_graph":{"description":"Inspect public graph dimensions and simulator state.","inputSchema":{"type":"object","properties":{}}},"propose_campaign_plan":{"description":"Search a risk-sensitive graph-MPPI campaign plan.","inputSchema":{"type":"object","properties":{"horizon":{"type":"integer","minimum":1,"maximum":6}}}},"simulate_intervention":{"description":"Generate multiple counterfactual graph futures.","inputSchema":{"type":"object","properties":{}}},"verify_intervention":{"description":"Apply CVaR/OOD/constraint verification before execution.","inputSchema":{"type":"object","properties":{}}}}
x_tools=GraphWorldAdsXTools(); print(x_tools.inspect())

## X14. Research comparison table and required ablations

Run the following before making research claims:

| Model | Graph | Semantics | Generative futures | Opponents | Planner | Risk gate |
|---|---|---|---|---|---|---|
| MLP | No | No | No | No | No | No |
| GraphSAGE baseline | Yes | Optional | No | No | No | No |
| deterministic relation GTN | Yes | Yes | No | No | random MPC | sigma |
| GraphWorld-Ads baseline | Yes | limited | Gaussian | centralized | random MPC | sigma |
| GraphWorld-Ads-X | dynamic | Yes | **flow matching** | **learned** | **MPPI** | **CVaR+OOD** |

### Primary metrics
- one-step dynamic-state RMSE;
- H-step rollout drift;
- energy/coverage calibration of generated futures;
- simulated advertiser value / ROAS;
- regret against a reduced-action simulator oracle;
- budget utilization and constraint violations;
- OPE SNIPS / doubly robust estimate / effective sample size;
- US+ES → JP zero-shot RMSE;
- abstention precision and coverage;
- planner action efficiency.

### Novelty claim discipline
Do **not** claim that Graph Transformers, flow matching, MPPI, optimal transport, GSP, or opponent modeling are individually novel. The research contribution to test is their **intervention-aware integration** for a heterogeneous advertising marketplace, with structural graph edits, multi-future world modeling, opponent responses, and risk-gated planning.

In [ ]:
# Final audited invariants: schema, graph, state coherence, structure persistence and split integrity.
assert ORIGINAL_BASELINE_CELLS==33
assert {"query_raw_id","product_raw_id","query_id","product_id","locale","label","rel"}.issubset(esci.columns)
assert esci.duplicated(["locale","query_id","product_id"]).sum()==0
assert node_sem.shape==(N,X_SEM_DIM) and torch.isfinite(node_sem).all()
assert edge_index.shape[0]==2 and edge_index.max().item()<N and edge_index.shape[1]==len(set(zip(edge_index[0].tolist(),edge_index[1].tolist(),edge_type.tolist())))
assert len(campaign_adv_idx)==C and x_dynamic_mask.shape==(N,DYN_DIM)
assert torch.isfinite(x_delta_mean).all() and torch.isfinite(x_delta_std).all()
assert base_eligibility.shape==rel_cq.shape==effective_rel_cq.shape==targetable_cq.shape
assert np.all(base_eligibility<=targetable_cq.astype(np.float32))
assert all(0<=int(k)<len(MARKETS) for k in campaign_locale) and all(0<=int(k)<len(MARKETS) for k in query_locale)
# Add-query must be capable of changing state for at least one campaign whenever a latent candidate exists.
latent=np.argwhere(targetable_cq&(base_eligibility==0))
if len(latent):
    ci,qj=map(int,latent[0]); xs=initial_x_state(); iv=Intervention(np.zeros(C,np.float32),np.zeros(C,np.float32),[(ci,qj)],[],[]); nx=apply_structural_intervention(xs,iv); assert xs.eligibility[ci,qj]==0 and nx.eligibility[ci,qj]==1
# Market expansion must persist.
if C and len(MARKETS)>1:
    ci=0; foreign=[m for m in MARKETS if market_to_k[m]!=campaign_locale[ci]]
    if foreign:
        xs=initial_x_state(); iv=Intervention(np.zeros(C,np.float32),np.zeros(C,np.float32),[],[],[(ci,foreign[0])]); nx=apply_structural_intervention(xs,iv); assert nx.market_access[ci,market_to_k[foreign[0]]]==1
# Deterministic controls and derived-state constraints.
r=x_train[0]; base=deterministic_feature_transition(r["x"],r["iv"]); zero_res=torch.zeros_like(r["x"]); proj=project_valid_state(r["x"],zero_res,r["iv"]); assert torch.allclose(proj[campaign_gids,0],base[campaign_gids,0]) and torch.allclose(proj[campaign_gids,2],base[campaign_gids,2]) and torch.allclose(proj[campaign_gids,8],base[campaign_gids,8]); assert torch.allclose(proj[query_gids,8],base[query_gids,8])
# Budget/spend coherence in simulator.
xs=initial_x_state(); iv=behavior_intervention(xs); nx=market_step_gsp(xs,iv,False); assert np.all(nx.base.spend<=nx.base.budget+1e-4) and np.all(nx.base.cum_revenue>=xs.base.cum_revenue)
# Structure-aware graph contains persistent targeting edges and finite weights.
g=build_intervention_graph(_zero_iv,xs); assert g["edge_index"].max().item()<g["n_ext"] and len(g["edge_type"])==g["edge_index"].shape[1]==len(g["edge_weight"]) and torch.isfinite(g["edge_weight"]).all()
# JP protocol truly withholds dynamic inputs/targets/stats while retaining static semantics.
if jp_input_mask.any():
    masked=mask_jp_input(x_train[0]["x"])
    for j in range(DYN_DIM):
        m=jp_input_mask[:,j]
        if m.any():
            pass
    assert not (train_us_es_mask & jp_input_mask).any()
assert not (train_us_es_mask & jp_target_mask).any()
assert (jp_target_mask <= jp_input_mask).all()
assert x_train_eps.isdisjoint(x_val_eps|x_test_eps) and x_val_eps.isdisjoint(x_test_eps)
print("GraphWorld-Ads-X audited wiring/state/schema checks: PASS")